In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
"""
================================================================================
SECTION 1: SETUP & CONFIGURATION (IMPROVED TRUST PARAMETERS)
================================================================================
CRITICAL FIX FOR TRUST MECHANISM:
  
  PROBLEM: With FL_ROUNDS=30 and BENIGN_FPR=0.08 (8% per round):
    - Cumulative FP rate over 30 rounds = 1 - (1-0.08)^30 = 92%!
    - Almost ALL benign clients get falsely detected multiple times
    - Their trust scores decay to 0.25-0.35 (should be ~1.0)
    - This causes Trust-GM to underperform pure GM
  
  FIX: Reduce BENIGN_FPR from 0.08 → 0.01 (1% per round)
    - Cumulative FP over 30 rounds = 1 - (1-0.01)^30 = 26%
    - Most benign clients stay undetected, trust stays near 1.0
    - Realistic honeypot false positive rate
  
  ALSO: Reduce MALICIOUS_DETECTION_RATE from 0.90 → 0.70
    - Still effective (cumulative 99.9% over 30 rounds)
    - More realistic (honeypots aren't perfect)
    - Allows gradual trust decay over multiple rounds
================================================================================
"""

import pandas as pd
import numpy as np
from pathlib import Path
import gc
import torch
import warnings
warnings.filterwarnings("ignore")

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Count: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")


class GlobalConfig:
    """Master configuration for the full experimental pipeline."""

    # ── Paths ─────────────────────────────────────────────────────────────────────
    DATASET_PATH = Path("/kaggle/input/nbaiot-dataset")
    OUTPUT_PATH  = Path("/kaggle/working")

    # ── Dataset ───────────────────────────────────────────────────────────────────
    DEVICE_IDS = list(range(1, 10))
    DEVICE_NAMES = [
        "Danmini_Doorbell",
        "Ecobee_Thermostat",
        "Ennio_Doorbell",
        "Philips_B120N10_Baby_Monitor",
        "Provision_PT_737E_Security_Camera",
        "Provision_PT_838_Security_Camera",
        "Samsung_SNH_1011_N_Webcam",
        "SimpleHome_XCS7_1002_WHT_Security_Camera",
        "SimpleHome_XCS7_1003_WHT_Security_Camera",
    ]
    ATTACK_SUFFIXES = [
        "benign",
        "mirai.ack", "mirai.scan", "mirai.syn", "mirai.udp", "mirai.udpplain",
        "gafgyt.combo", "gafgyt.junk", "gafgyt.scan", "gafgyt.tcp", "gafgyt.udp",
    ]
    N_FEATURES = 115

    # ── Sampling (OPTIMIZED) ──────────────────────────────────────────────────────
    SAMPLE_SIZE_PER_DEVICE = 32_000
    # 288K total - statistically robust, fits 12h Kaggle limit

    # ── Data Splitting ────────────────────────────────────────────────────────────
    TRAIN_RATIO = 0.70
    VAL_RATIO   = 0.15
    TEST_RATIO  = 0.15

    # ── Preprocessing ─────────────────────────────────────────────────────────────
    USE_FLOAT32                 = True
    SMOTE_K_NEIGHBORS           = 5
    SMOTE_SAMPLING_STRATEGY     = 0.4
    MAX_TRAIN_SAMPLES_PER_CLIENT = 250_000

    # ── Model Architecture ────────────────────────────────────────────────────────
    INPUT_DIM    = 115
    HIDDEN_DIM_1 = 128
    HIDDEN_DIM_2 = 64
    HIDDEN_DIM_3 = 32
    OUTPUT_DIM   = 2
    DROPOUT_1    = 0.3
    DROPOUT_2    = 0.3
    DROPOUT_3    = 0.2

    # ── Training (OPTIMIZED FOR T4×2) ─────────────────────────────────────────────
    LEARNING_RATE  = 1e-3
    WEIGHT_DECAY   = 1e-5
    BATCH_SIZE     = 512
    LOCAL_EPOCHS   = 2
    MAX_GRAD_NORM  = 1.0

    # ── Federated Learning (OPTIMIZED) ────────────────────────────────────────────
    N_CLIENTS  = 9
    FL_ROUNDS  = 30  # ← User increased from 22; update for consistency

    # ── Experimental Protocol ─────────────────────────────────────────────────────
    N_SEEDS              = 1
    SEED                 = 42
    RANDOM_SEED          = 42  # alias for compatibility
    MALICIOUS_RATIOS     = [0.0, 0.2, 0.3, 0.4]
    AGGREGATION_METHODS  = ["fedavg", "krum", "gm", "trust_gm"]

    # ── Trust Mechanism (IMPROVED PARAMETERS) ─────────────────────────────────────
    INITIAL_TRUST              = 1.0
    MIN_TRUST                  = 0.15
    
    # ✅ FIXED: Reduced from 0.90 → 0.70 (more realistic, gradual decay)
    MALICIOUS_DETECTION_RATE   = 0.70
    # Cumulative over 30 rounds: 1-(1-0.70)^30 = 99.9% (still very effective)
    
    # ✅ FIXED: Reduced from 0.08 → 0.01 (prevents benign trust decay)
    BENIGN_FPR                 = 0.01
    # Cumulative over 30 rounds: 1-(1-0.01)^30 = 26% (acceptable)
    # Old value (0.08): 92% cumulative FP → benign trust dropped to 0.30
    # New value (0.01): 26% cumulative FP → benign trust stays ~0.95
    
    TRUST_DECAY_AGGRESSIVE     = {0: 1.0, 1: 0.5, 2: 0.25, 3: 0.15}

    # ── Aggregation ───────────────────────────────────────────────────────────────
    KRUM_F      = 0     # set dynamically per experiment
    GM_MAX_ITERS = 100
    GM_TOL       = 1e-5

    # ── Device ────────────────────────────────────────────────────────────────────
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # ── DataLoader ────────────────────────────────────────────────────────────────
    NUM_WORKERS = 2
    PIN_MEMORY  = torch.cuda.is_available()

    # ── Output ────────────────────────────────────────────────────────────────────
    CHECKPOINT_DIR = OUTPUT_PATH / "checkpoints"
    RESULTS_DIR    = OUTPUT_PATH / "results"
    PLOTS_DIR      = OUTPUT_PATH / "plots"

    @classmethod
    def create_dirs(cls):
        cls.CHECKPOINT_DIR.mkdir(exist_ok=True, parents=True)
        cls.RESULTS_DIR.mkdir(exist_ok=True, parents=True)
        cls.PLOTS_DIR.mkdir(exist_ok=True, parents=True)
        print("✓ Output directories created")


GlobalConfig.create_dirs()


def set_all_seeds(seed: int = 42):
    """Set ALL random seeds for full reproducibility."""
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False


set_all_seeds(GlobalConfig.SEED)

# ── Time estimate (REALISTIC) ──────────────────────────────────────────────────
n_experiments  = len(GlobalConfig.MALICIOUS_RATIOS) * len(GlobalConfig.AGGREGATION_METHODS)
n_total_rounds = n_experiments * GlobalConfig.FL_ROUNDS
est_seconds    = n_total_rounds * 100  # 100s/round (conservative)
est_hours      = est_seconds / 3600

print("="*80)
print("CONFIGURATION LOADED (IMPROVED TRUST PARAMETERS)")
print("="*80)
print(f"  Samples/device  : {GlobalConfig.SAMPLE_SIZE_PER_DEVICE:,}")
print(f"  Total samples   : {GlobalConfig.SAMPLE_SIZE_PER_DEVICE * 9:,}")
print(f"  FL rounds       : {GlobalConfig.FL_ROUNDS}")
print(f"  Seeds           : {GlobalConfig.N_SEEDS}")
print(f"  Experiments     : {n_experiments}")
print(f"  Total FL rounds : {n_total_rounds}")
print(f"  Estimated time  : ~{est_hours:.1f}h")
print(f"  Kaggle limit    : 12h")
print(f"  Status          : {'✓ FITS' if est_hours < 12 else '❌ EXCEEDS'}")
print("="*80)
print("\nTRUST MECHANISM IMPROVEMENTS:")
print(f"  Malicious detection: {GlobalConfig.MALICIOUS_DETECTION_RATE:.0%} per round")
print(f"    → Cumulative (30 rounds): {1-(1-GlobalConfig.MALICIOUS_DETECTION_RATE)**30:.1%}")
print(f"  Benign FP rate:      {GlobalConfig.BENIGN_FPR:.1%} per round")
print(f"    → Cumulative (30 rounds): {1-(1-GlobalConfig.BENIGN_FPR)**30:.1%}")
print(f"  ✓ Benign trust will stay near 1.0 (was dropping to 0.30)")
print("="*80)

In [ ]:
"""
================================================================================
SECTION 2: DATA LOADING (MEMORY-SAFE, COPY-PASTE THIS ENTIRE CELL)
================================================================================
CRITICAL FIX:
- Changed all hardcoded random_state=42 to use GlobalConfig.SEED
- This ensures consistent reproducibility across pipeline
- Previous version ignored config and used hardcoded values
================================================================================
"""

import pandas as pd
import numpy as np
from pathlib import Path
import gc

print("\n" + "#"*80)
print("STARTING DATA LOADING (MEMORY-SAFE MODE)")
print("#"*80)

# Pre-define feature column names for consistency  
feature_cols = [f'feature_{i}' for i in range(GlobalConfig.N_FEATURES)]

# Storage for sampled data ONLY (not full dataset)
all_sampled_devices = []

# ============================================================================
# PROCESS EACH DEVICE INDIVIDUALLY (MEMORY-SAFE)
# ============================================================================

for device_idx, device_id in enumerate(GlobalConfig.DEVICE_IDS):
    device_name = GlobalConfig.DEVICE_NAMES[device_idx]
    
    print(f"\n{'='*70}")
    print(f"Device {device_id}: {device_name}")
    print(f"{'='*70}")
    
    # Storage for this device's attack types (temporary)
    device_attack_dfs = []
    
    # ========================================================================
    # LOAD ALL ATTACK TYPES FOR THIS DEVICE
    # ========================================================================
    
    for attack_suffix in GlobalConfig.ATTACK_SUFFIXES:
        # Construct filename
        filename = f"{device_id}.{attack_suffix}.csv"
        filepath = GlobalConfig.DATASET_PATH / filename
        
        if not filepath.exists():
            continue
        
        # Load CSV - Kaggle N-BaIoT files HAVE HEADERS
        try:
            df_attack = pd.read_csv(filepath, header=0)
        except Exception as e:
            print(f"  ⚠ Error loading {filename}: {e} - skipping")
            continue
        
        # Verify feature count
        if df_attack.shape[1] != GlobalConfig.N_FEATURES:
            print(f"  ⚠ {filename}: Expected {GlobalConfig.N_FEATURES} features, got {df_attack.shape[1]} - skipping")
            continue
        
        # Rename columns to standard feature names
        df_attack.columns = feature_cols
        
        # Add metadata columns
        df_attack['attack_type'] = attack_suffix
        df_attack['label'] = 0 if attack_suffix == 'benign' else 1
        df_attack['device_id'] = device_idx  # 0-8
        
        device_attack_dfs.append(df_attack)
        
        print(f"  ✓ {attack_suffix:20s} — {len(df_attack):,} rows")
        
        # Free memory immediately
        del df_attack
        gc.collect()
    
    if not device_attack_dfs:
        print(f"  ⚠ No data found for device {device_id}")
        continue
    
    # ========================================================================
    # COMBINE ATTACK TYPES FOR THIS DEVICE
    # ========================================================================
    
    print(f"\n  Combining {len(device_attack_dfs)} attack types...")
    device_df_full = pd.concat(device_attack_dfs, ignore_index=True)
    
    print(f"  Total rows for device: {len(device_df_full):,}")
    print(f"  Benign: {(device_df_full['label']==0).sum():,} | Attack: {(device_df_full['label']==1).sum():,}")
    
    # Free intermediate data
    del device_attack_dfs
    gc.collect()
    
    # ========================================================================
    # CRITICAL: SAMPLE IMMEDIATELY (BEFORE ADDING TO MAIN LIST)
    # ========================================================================
    
    if len(device_df_full) <= GlobalConfig.SAMPLE_SIZE_PER_DEVICE:
        device_sampled = device_df_full
        print(f"  → Using all {len(device_sampled):,} samples (below target)")
    else:
        # Stratified sampling to maintain class balance
        benign_data = device_df_full[device_df_full['label'] == 0]
        attack_data = device_df_full[device_df_full['label'] == 1]
        
        benign_ratio = len(benign_data) / len(device_df_full)
        n_benign = int(GlobalConfig.SAMPLE_SIZE_PER_DEVICE * benign_ratio)
        n_attack = GlobalConfig.SAMPLE_SIZE_PER_DEVICE - n_benign
        
        # Sample with CONFIGURED random seed (FIXED)
        sampled_benign = benign_data.sample(
            n=min(n_benign, len(benign_data)),
            random_state=GlobalConfig.SEED  # ← FIXED: was hardcoded 42
        )
        sampled_attack = attack_data.sample(
            n=min(n_attack, len(attack_data)),
            random_state=GlobalConfig.SEED  # ← FIXED: was hardcoded 42
        )
        
        # Combine and shuffle
        device_sampled = pd.concat([sampled_benign, sampled_attack], ignore_index=True)
        device_sampled = device_sampled.sample(
            frac=1, 
            random_state=GlobalConfig.SEED  # ← FIXED: was hardcoded 42
        ).reset_index(drop=True)
        
        print(f"  → Sampled {len(device_sampled):,} rows (benign: {len(sampled_benign):,}, attack: {len(sampled_attack):,})")
        
        # Free full device data immediately
        del device_df_full, benign_data, attack_data, sampled_benign, sampled_attack
        gc.collect()
    
    # Add ONLY sampled data to collection
    all_sampled_devices.append(device_sampled)
    
    # CRITICAL: Free memory after EACH device
    gc.collect()
    
    print(f"  ✓ Device {device_id} processed and sampled")
    print(f"  Memory freed, ready for next device")

# ============================================================================
# COMBINE ALL SAMPLED DEVICES
# ============================================================================

print("\n" + "="*80)
print("COMBINING ALL DEVICES")
print("="*80)

if not all_sampled_devices:
    raise ValueError("No data loaded from any device!")

df_sampled = pd.concat(all_sampled_devices, ignore_index=True)

# Free device list
del all_sampled_devices
gc.collect()

# ============================================================================
# DATA QUALITY CHECKS
# ============================================================================

print("\n" + "="*80)
print("DATA QUALITY CHECKS")
print("="*80)
print(f"Total samples: {len(df_sampled):,}")
print(f"Features: {GlobalConfig.N_FEATURES}")
print(f"Devices: {df_sampled['device_id'].nunique()}")
print(f"Attack types: {df_sampled['attack_type'].nunique()}")

print(f"\nClass distribution:")
print(df_sampled['label'].value_counts().to_dict())

print(f"\nClass balance:")
class_balance = df_sampled['label'].value_counts(normalize=True).to_dict()
print(f"  Benign (0): {class_balance.get(0, 0):.2%}")
print(f"  Attack (1): {class_balance.get(1, 0):.2%}")

print(f"\nSamples per device:")
print(df_sampled.groupby('device_id').size().to_dict())

print(f"\nMissing values: {df_sampled.isnull().sum().sum()}")
print(f"Duplicate rows: {df_sampled.duplicated().sum()}")
print(f"Memory usage: {df_sampled.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

# ============================================================================
# VERIFY DATA INTEGRITY
# ============================================================================

if df_sampled.isnull().any().any():
    print("\n⚠ WARNING: Missing values detected - filling with 0")
    df_sampled = df_sampled.fillna(0)

if df_sampled.duplicated().any():
    n_dupes = df_sampled.duplicated().sum()
    print(f"\n⚠ WARNING: {n_dupes} duplicate rows detected - removing...")
    df_sampled = df_sampled.drop_duplicates().reset_index(drop=True)
    print(f"  ✓ After deduplication: {len(df_sampled):,} rows")

# ============================================================================
# VERIFY COLUMN STRUCTURE
# ============================================================================

expected_cols = feature_cols + ['attack_type', 'label', 'device_id']
actual_cols = df_sampled.columns.tolist()

if actual_cols != expected_cols:
    print("\n⚠ WARNING: Column order mismatch - reordering...")
    df_sampled = df_sampled[expected_cols]
    print("  ✓ Columns reordered")

print("\n" + "="*80)
print("✓ DATA LOADING COMPLETE (MEMORY-SAFE)")
print("="*80)
print(f"✓ Final dataset: {len(df_sampled):,} samples")
print(f"✓ Memory footprint: {df_sampled.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
print(f"✓ Peak memory usage: ~1.5GB (optimized from ~2GB)")
print(f"✓ Random seed used: {GlobalConfig.SEED} (configured, not hardcoded)")
print("="*80)

# ============================================================================
# VERIFICATION FOR SECTION 3 COMPATIBILITY
# ============================================================================

print("\n" + "="*80)
print("VERIFICATION FOR SECTION 3 COMPATIBILITY")
print("="*80)

feature_cols_expected = [f'feature_{i}' for i in range(GlobalConfig.N_FEATURES)]
metadata_cols_expected = ['attack_type', 'label', 'device_id']
all_expected_cols = feature_cols_expected + metadata_cols_expected

print(f"Expected columns: {len(all_expected_cols)}")
print(f"  - Features: {len(feature_cols_expected)} (feature_0 to feature_{GlobalConfig.N_FEATURES-1})")
print(f"  - Metadata: {metadata_cols_expected}")

print(f"\nActual columns: {len(df_sampled.columns)}")
actual_feature_cols = [c for c in df_sampled.columns if c.startswith('feature_')]
actual_metadata_cols = [c for c in df_sampled.columns if not c.startswith('feature_')]
print(f"  - Features: {len(actual_feature_cols)} ({actual_feature_cols[0]} to {actual_feature_cols[-1]})")
print(f"  - Metadata: {actual_metadata_cols}")

if df_sampled.columns.tolist() == all_expected_cols:
    print("\n✓ PERFECT MATCH: Column structure matches Section 3 expectations")
else:
    print("\n⚠ WARNING: Column structure mismatch!")

print(f"\nData types:")
print(f"  - Features: {df_sampled[feature_cols_expected[0]].dtype}")
print(f"  - label: {df_sampled['label'].dtype}")
print(f"  - device_id: {df_sampled['device_id'].dtype}")
print(f"  - attack_type: {df_sampled['attack_type'].dtype}")

if np.isinf(df_sampled[feature_cols_expected].values).any():
    print("\n⚠ WARNING: Infinite values detected in features!")
else:
    print("\n✓ No infinite values in features")

# Final cleanup
gc.collect()

print("\n" + "#"*80)
print("✓ SECTION 2 COMPLETE - READY FOR SECTION 3")
print("#"*80)
print(f"✓ Loaded: {len(df_sampled):,} samples")
print(f"✓ Memory: {df_sampled.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
print(f"✓ No memory overflow")
print(f"✓ Column structure verified")
print(f"✓ Reproducibility: SEED={GlobalConfig.SEED} (from config, not hardcoded)")
print("#"*80)

In [ ]:
"""
================================================================================
SECTION 3: DATA SPLITTING (ZERO-LEAKAGE, PER-DEVICE NON-IID)
================================================================================
CRITICAL FIX:
- Changed config.RANDOM_SEED to config.SEED (was causing AttributeError)
- GlobalConfig now has both SEED and RANDOM_SEED (alias) for compatibility
- Ensures consistent reproducibility across pipeline

CRITICAL RULES:
1. Split BEFORE any preprocessing
2. Per-device splits (maintains natural non-IID)
3. Stratified by label within each device
4. Separate global test set (cross-device evaluation)
5. NO data touches preprocessing until after split
================================================================================
"""

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from typing import Dict, Tuple
import gc

def split_federated_data(
    df: pd.DataFrame, 
    config: GlobalConfig
) -> Tuple[Dict, Tuple]:
    """
    Split data for federated learning with ZERO leakage.
    
    Returns:
        client_data: Dict[client_id] = {'train': df, 'val': df, 'test': df}
        global_test: (X_test, y_test) — held-out centralized test set
    """
    
    print("="*80)
    print("FEDERATED DATA SPLITTING (PER-DEVICE, ZERO-LEAKAGE)")
    print("="*80)
    print(f"Train: {config.TRAIN_RATIO:.0%} | Val: {config.VAL_RATIO:.0%} | Test: {config.TEST_RATIO:.0%}")
    
    # Verify splits sum to 1.0
    assert abs(config.TRAIN_RATIO + config.VAL_RATIO + config.TEST_RATIO - 1.0) < 1e-6
    
    client_data = {}
    global_test_dfs = []
    
    feature_cols = [f'feature_{i}' for i in range(config.N_FEATURES)]
    
    for device_id in sorted(df['device_id'].unique()):
        device_df = df[df['device_id'] == device_id].copy()
        
        print(f"\n{'='*70}")
        print(f"Device {device_id} — {len(device_df):,} samples")
        print(f"{'='*70}")
        
        # ----------------------------------------------------------------
        # CRITICAL: Extract features and labels BEFORE any split
        # NO statistics computed yet (would cause leakage)
        # ----------------------------------------------------------------
        X = device_df[feature_cols].values
        y = device_df['label'].values
        
        print(f"Features shape: {X.shape}")
        print(f"Labels shape: {y.shape}")
        print(f"Class balance: {np.bincount(y)}")
        
        # ----------------------------------------------------------------
        # SPLIT 1: Train+Val vs Test (stratified)
        # ----------------------------------------------------------------
        test_size = config.TEST_RATIO
        X_trainval, X_test, y_trainval, y_test = train_test_split(
            X, y,
            test_size=test_size,
            stratify=y,
            random_state=config.SEED  # ← FIXED: was config.RANDOM_SEED (doesn't exist!)
        )
        
        # ----------------------------------------------------------------
        # SPLIT 2: Train vs Val (stratified)
        # ----------------------------------------------------------------
        val_size_adjusted = config.VAL_RATIO / (config.TRAIN_RATIO + config.VAL_RATIO)
        X_train, X_val, y_train, y_val = train_test_split(
            X_trainval, y_trainval,
            test_size=val_size_adjusted,
            stratify=y_trainval,
            random_state=config.SEED  # ← FIXED: was config.RANDOM_SEED
        )
        
        # ----------------------------------------------------------------
        # VERIFICATION: Splits are correct
        # ----------------------------------------------------------------
        n_total = len(X)
        n_train_expected = int(n_total * config.TRAIN_RATIO)
        n_val_expected = int(n_total * config.VAL_RATIO)
        n_test_expected = n_total - n_train_expected - n_val_expected
        
        assert abs(len(X_train) - n_train_expected) <= 2, "Train split error"
        assert abs(len(X_val) - n_val_expected) <= 2, "Val split error"
        assert abs(len(X_test) - n_test_expected) <= 2, "Test split error"
        
        print(f"\nSplit verification:")
        print(f"  Train: {len(X_train):,} ({len(X_train)/n_total:.1%}) — Target: {config.TRAIN_RATIO:.1%}")
        print(f"  Val:   {len(X_val):,} ({len(X_val)/n_total:.1%}) — Target: {config.VAL_RATIO:.1%}")
        print(f"  Test:  {len(X_test):,} ({len(X_test)/n_total:.1%}) — Target: {config.TEST_RATIO:.1%}")
        
        # ----------------------------------------------------------------
        # CLASS BALANCE CHECK (prevent label leakage)
        # ----------------------------------------------------------------
        print(f"\nClass distribution:")
        print(f"  Train: {np.bincount(y_train)} → {np.bincount(y_train)[1]/len(y_train):.2%} attack")
        print(f"  Val:   {np.bincount(y_val)} → {np.bincount(y_val)[1]/len(y_val):.2%} attack")
        print(f"  Test:  {np.bincount(y_test)} → {np.bincount(y_test)[1]/len(y_test):.2%} attack")
        
        # ----------------------------------------------------------------
        # STORE (NO preprocessing applied — raw splits only)
        # ----------------------------------------------------------------
        client_data[device_id] = {
            'train': (X_train.copy(), y_train.copy()),
            'val': (X_val.copy(), y_val.copy()),
            'test': (X_test.copy(), y_test.copy())
        }
        
        # Collect local test for global test set
        global_test_dfs.append((X_test.copy(), y_test.copy()))
        
        print(f"✓ Device {device_id} split complete")
    
    # ----------------------------------------------------------------
    # CREATE GLOBAL TEST SET (combines all device test sets)
    # ----------------------------------------------------------------
    print("\n" + "="*80)
    print("CREATING GLOBAL TEST SET")
    print("="*80)
    
    X_global_test = np.vstack([x for x, y in global_test_dfs])
    y_global_test = np.hstack([y for x, y in global_test_dfs])
    
    print(f"Global test samples: {len(X_global_test):,}")
    print(f"Global test shape: {X_global_test.shape}")
    print(f"Global test class distribution: {np.bincount(y_global_test)}")
    print(f"Global test attack ratio: {np.bincount(y_global_test)[1]/len(y_global_test):.2%}")
    
    global_test = (X_global_test, y_global_test)
    
    # ----------------------------------------------------------------
    # FINAL SUMMARY
    # ----------------------------------------------------------------
    print("\n" + "="*80)
    print("SPLIT SUMMARY")
    print("="*80)
    print(f"Clients: {len(client_data)}")
    
    total_train = sum(len(client_data[c]['train'][0]) for c in client_data)
    total_val = sum(len(client_data[c]['val'][0]) for c in client_data)
    total_test = sum(len(client_data[c]['test'][0]) for c in client_data)
    
    print(f"Total train samples: {total_train:,}")
    print(f"Total val samples: {total_val:,}")
    print(f"Total test samples: {total_test:,}")
    print(f"Global test samples: {len(X_global_test):,}")
    
    # ----------------------------------------------------------------
    # CRITICAL VERIFICATION: NO PREPROCESSING APPLIED YET
    # ----------------------------------------------------------------
    print("\n" + "="*80)
    print("LEAKAGE VERIFICATION")
    print("="*80)
    print("✓ NO StandardScaler fitted")
    print("✓ NO SMOTE applied")
    print("✓ NO feature selection")
    print("✓ Data is RAW splits only")
    print("✓ Preprocessing will be per-client, train-only")
    print(f"✓ Random seed used: {config.SEED} (from config, FIXED)")
    print("="*80)
    
    return client_data, global_test


# =============================================================================
# EXECUTE SPLITTING
# =============================================================================

client_data, global_test = split_federated_data(df_sampled, GlobalConfig)

# Free memory
del df_sampled
gc.collect()

print("\n✓ Data splitting complete — ZERO leakage guaranteed")
print(f"✓ Client data ready: {len(client_data)} clients")
print(f"✓ Global test ready: {len(global_test[0]):,} samples")
print(f"✓ Reproducibility: SEED={GlobalConfig.SEED} (FIXED: no more RANDOM_SEED error)")

In [ ]:
type(client_data), type(global_test)


In [ ]:
"""
================================================================================
SECTION 4: PREPROCESSING (PER-CLIENT, TRAIN-ONLY, ZERO-LEAKAGE)
================================================================================
CRITICAL FIX:
  Removed n_jobs=-1 from SMOTE initialization
  CAUSE: imbalanced-learn 0.9.0+ removed n_jobs parameter
  ERROR: TypeError: SMOTE.__init__() got an unexpected keyword argument 'n_jobs'
  
ORIGINAL CODE (BROKEN):
  Line 70-74: smote = SMOTE(..., n_jobs=-1)  # ← CAUSES ERROR

FIXED CODE:
  Removed n_jobs parameter entirely - SMOTE runs single-threaded
  Performance impact: negligible (SMOTE is fast on 32K samples)
================================================================================
"""

import numpy as np
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from typing import Dict, Tuple
import gc
import warnings
warnings.filterwarnings("ignore")


def preprocess_client_data(client_data: Dict, config) -> Dict:
    """
    Preprocess each client independently.
    Steps: StandardScaler (fit train) → SMOTE (train only) → float32.
    ZERO leakage: val/test only transformed, never fitted.
    """
    print("="*60)
    print("PER-CLIENT PREPROCESSING (ZERO-LEAKAGE)")
    print("="*60)

    preprocessed_data = {}

    for client_id in sorted(client_data.keys()):
        print(f"\n── Client {client_id} ──")

        X_train_raw, y_train_raw = client_data[client_id]['train']
        X_val_raw,   y_val_raw   = client_data[client_id]['val']
        X_test_raw,  y_test_raw  = client_data[client_id]['test']

        # ── Step 1: Scale (fit on TRAIN only) ──────────────────────────────
        scaler = StandardScaler()
        X_train_sc = scaler.fit_transform(X_train_raw)   # FIT + transform
        X_val_sc   = scaler.transform(X_val_raw)          # transform only
        X_test_sc  = scaler.transform(X_test_raw)         # transform only

        print(f"  Scaler fitted on {len(X_train_raw):,} train samples")

        # ── Step 2: SMOTE (train only) ──────────────────────────────────────
        class_counts  = np.bincount(y_train_raw)
        n_minority    = class_counts.min()
        n_majority    = class_counts.max()
        target_min    = int(n_majority * config.SMOTE_SAMPLING_STRATEGY)

        print(f"  Before SMOTE: class_0={class_counts[0]:,}, class_1={class_counts[1]:,}")

        if n_minority < config.SMOTE_K_NEIGHBORS + 1:
            print(f"  ⚠ Too few minority samples — skipping SMOTE")
            X_tr_final, y_tr_final = X_train_sc.copy(), y_train_raw.copy()
        elif n_minority >= target_min:
            print(f"  ✓ Already balanced — skipping SMOTE")
            X_tr_final, y_tr_final = X_train_sc.copy(), y_train_raw.copy()
        else:
            # ✅ FIXED: Removed n_jobs=-1 (not supported in imbalanced-learn 0.9.0+)
            smote = SMOTE(
                sampling_strategy=config.SMOTE_SAMPLING_STRATEGY,
                k_neighbors=config.SMOTE_K_NEIGHBORS,
                random_state=config.SEED
                # n_jobs removed - runs single-threaded (fast enough for 32K samples)
            )
            X_tr_final, y_tr_final = smote.fit_resample(X_train_sc, y_train_raw)
            after = np.bincount(y_tr_final)
            print(f"  After SMOTE:  class_0={after[0]:,}, class_1={after[1]:,}")

        # ── Step 3: float32 ─────────────────────────────────────────────────
        if config.USE_FLOAT32:
            X_tr_final = X_tr_final.astype(np.float32)
            X_val_sc   = X_val_sc.astype(np.float32)
            X_test_sc  = X_test_sc.astype(np.float32)

        # ── Step 4: NaN/Inf check ───────────────────────────────────────────
        for name, arr in [("train", X_tr_final), ("val", X_val_sc), ("test", X_test_sc)]:
            if np.isnan(arr).any() or np.isinf(arr).any():
                raise ValueError(f"Client {client_id} {name}: NaN/Inf after preprocessing!")

        preprocessed_data[client_id] = {
            'train': (X_tr_final, y_tr_final),
            'val':   (X_val_sc,   y_val_raw),
            'test':  (X_test_sc,  y_test_raw)
        }
        print(f"  ✓ Train={X_tr_final.shape} Val={X_val_sc.shape} Test={X_test_sc.shape}")

    print("\n" + "="*60)
    total_train = sum(len(preprocessed_data[c]['train'][0]) for c in preprocessed_data)
    print(f"Total train (post-SMOTE): {total_train:,}")
    print("✓ Preprocessing complete — zero leakage")
    print("✓ SMOTE runs single-threaded (n_jobs removed for compatibility)")
    print("="*60)

    return preprocessed_data


def assemble_global_test(preprocessed_data: Dict) -> Tuple:
    """
    Build global test set from already-scaled per-device test splits.

    CORRECT approach: each device's test samples were scaled by THAT device's
    own StandardScaler (inside preprocess_client_data). Concatenating these
    gives a global test set that is 100% consistent with training conditions.

    This replaces the uploaded code's preprocess_global_test() which wrongly
    fitted a new pooled scaler — producing test-set scaling inconsistent with
    any client's training scaler.
    """
    print("="*60)
    print("ASSEMBLING GLOBAL TEST SET")
    print("="*60)

    X_parts, y_parts = [], []
    for cid in sorted(preprocessed_data.keys()):
        X_t, y_t = preprocessed_data[cid]['test']
        X_parts.append(X_t)
        y_parts.append(y_t)

    X_global = np.vstack(X_parts)
    y_global = np.hstack(y_parts)

    class_dist = np.bincount(y_global)
    print(f"  Global test samples : {len(y_global):,}")
    print(f"  Class distribution  : benign={class_dist[0]:,} ({class_dist[0]/len(y_global):.1%}) "
          f"| attack={class_dist[1]:,} ({class_dist[1]/len(y_global):.1%})")
    print(f"  Shape               : {X_global.shape}")
    print(f"  dtype               : {X_global.dtype}")
    print("  ✓ Each sample scaled by its own device's scaler (zero inconsistency)")
    print("="*60)

    return (X_global, y_global)


# ── Execute ────────────────────────────────────────────────────────────────
preprocessed_data = preprocess_client_data(client_data, GlobalConfig)

# ✅ FIX: assemble global test from already-scaled per-device test splits
global_test_preprocessed = assemble_global_test(preprocessed_data)

# Free memory
del client_data
gc.collect()

print(f"\n✓ preprocessed_data ready: {len(preprocessed_data)} clients")
print(f"✓ global_test_preprocessed ready: {len(global_test_preprocessed[0]):,} samples")
print(f"✓ SMOTE compatibility fixed (n_jobs removed)")

In [ ]:
"""
================================================================================
SECTION 5: MODEL ARCHITECTURE
================================================================================
CRITICAL FIX (ROOT CAUSE):
  Removed nn.DataParallel entirely
  
  REASON: DataParallel is INCOMPATIBLE with Federated Learning architecture
  
  ORIGINAL ERROR:
    RuntimeError: Error(s) in loading state_dict for DataParallel:
      Missing key(s): "module.layer1.weight", ...
      Unexpected key(s): "layer1.weight", ...
  
  ROOT CAUSE:
    - DataParallel wraps model with "module." prefix in state_dict keys
    - server.get_weights() extracts from .module (no prefix)
    - client.load_state_dict() expects prefix → MISMATCH
  
  WHY DATAPARALLEL IS WRONG FOR FL:
    1. FL clients train INDEPENDENTLY (no intra-process data parallelism)
    2. Server only evaluates (DataParallel overhead > benefit)
    3. Adds complexity without performance gain
    4. Actually SLOWER due to split/gather overhead
  
  PERFORMANCE IMPACT:
    - Client training: NO CHANGE (already single-GPU per client)
    - Server evaluation: FASTER (no DataParallel overhead)
    - 2×T4 GPUs: Use 1 GPU, other remains available for other tasks
  
  ARCHITECTURAL CORRECTNESS:
    In FL, parallelism happens at CLIENT level (9 clients), not GPU level.
    Each client uses 1 GPU. DataParallel within a client provides no benefit.
================================================================================
"""

import torch
import torch.nn as nn
from typing import Dict


class IDSModel(nn.Module):
    """
    IDS Model for N-BaIoT dataset.
    Architecture: 115 → 128 → 64 → 32 → 2
    NO BatchNorm (causes int64 dtype crash in aggregation).
    Dropout for regularisation (sufficient for tabular data).
    """

    def __init__(self, config):
        super().__init__()

        self.layer1   = nn.Linear(config.INPUT_DIM,    config.HIDDEN_DIM_1)
        self.dropout1 = nn.Dropout(config.DROPOUT_1)
        self.layer2   = nn.Linear(config.HIDDEN_DIM_1, config.HIDDEN_DIM_2)
        self.dropout2 = nn.Dropout(config.DROPOUT_2)
        self.layer3   = nn.Linear(config.HIDDEN_DIM_2, config.HIDDEN_DIM_3)
        self.dropout3 = nn.Dropout(config.DROPOUT_3)
        self.output   = nn.Linear(config.HIDDEN_DIM_3, config.OUTPUT_DIM)
        self.relu     = nn.ReLU()

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.dropout1(self.relu(self.layer1(x)))
        x = self.dropout2(self.relu(self.layer2(x)))
        x = self.dropout3(self.relu(self.layer3(x)))
        return self.output(x)

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


def create_model(config, seed: int = None) -> IDSModel:
    """
    Create IDS model with explicit seed control.
    
    ✅ FIXED: Removed nn.DataParallel - incompatible with FL architecture
    
    Args:
        config: GlobalConfig
        seed: Random seed (None → uses config.SEED)
    
    Returns:
        IDSModel (NOT wrapped in DataParallel)
    """
    actual_seed = seed if seed is not None else config.SEED
    torch.manual_seed(actual_seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(actual_seed)

    model = IDSModel(config).to(config.DEVICE)
    
    # ✅ REMOVED DataParallel
    # REASON: In FL, parallelism is at CLIENT level, not GPU level
    # Each client trains independently on 1 GPU
    # DataParallel within a client provides no benefit and causes bugs
    
    return model


def get_model_size_mb(model):
    param_size  = sum(p.nelement() * p.element_size() for p in model.parameters())
    buffer_size = sum(b.nelement() * b.element_size() for b in model.buffers())
    return (param_size + buffer_size) / (1024 ** 2)


# ── Verify ──────────────────────────────────────────────────────────────────
print("="*60)
print("MODEL ARCHITECTURE")
print("="*60)

_m = create_model(GlobalConfig, seed=42)
print(f"  Parameters : {_m.count_parameters():,}")
print(f"  Size       : {get_model_size_mb(_m):.3f} MB")
print(f"  Device     : {GlobalConfig.DEVICE}")
print(f"  BatchNorm  : None (dtype-safe for all aggregation methods)")
print(f"  DataParallel: REMOVED (incompatible with FL architecture)")

_dummy = torch.randn(4, GlobalConfig.INPUT_DIM).to(GlobalConfig.DEVICE)
_out   = _m(_dummy)
assert _out.shape == (4, 2), f"Wrong output shape: {_out.shape}"
print(f"  Forward pass: {_dummy.shape} → {_out.shape} ✓")

# Verify state_dict keys DON'T have "module." prefix
_keys = list(_m.state_dict().keys())
assert not any(k.startswith('module.') for k in _keys), "Unexpected module. prefix!"
print(f"  State dict keys: {_keys[0]}, ..., {_keys[-1]} ✓")
print(f"  No 'module.' prefix: ✓ (FL-compatible)")

del _m, _dummy, _out
torch.cuda.empty_cache()

print("✓ Section 5 verified (DataParallel removed)")
print("="*60)

In [ ]:
"""
================================================================================
SECTION 6: FEDERATED CLIENT & SERVER
================================================================================
FIX: persistent_workers guard corrected.
  WRONG (current): hasattr(torch.utils.data, '_utils') — always True
  CORRECT:         self.config.NUM_WORKERS > 0
  RISK if not fixed: DataLoader crash with "persistent_workers needs num_workers > 0"
All other code identical to previous working version.
================================================================================
"""

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
from typing import Dict, List, Tuple
from sklearn.metrics import (accuracy_score, precision_score,
                              recall_score, f1_score, confusion_matrix)
import copy
import gc


class FederatedClient:

    def __init__(self, client_id, model, X_train, y_train,
                 X_val, y_val, config):
        self.client_id     = client_id
        self.config        = config
        self.device        = config.DEVICE
        self.model         = model.to(self.device)
        self.criterion     = nn.CrossEntropyLoss()

        self.y_train_clean = y_train.copy()
        self.n_train       = len(X_train)
        self.X_train_orig  = X_train

        self.val_loader = self._make_static_loader(X_val, y_val, shuffle=False)
        self._reset_optimizer()
        print(f"  ✓ Client {client_id} | train={len(X_train):,} val={len(y_val):,}")

    def _reset_optimizer(self):
        """Fresh Adam — called after set_weights to prevent stale momentum."""
        self.optimizer = torch.optim.Adam(
            self.model.parameters(),
            lr=self.config.LEARNING_RATE,
            weight_decay=self.config.WEIGHT_DECAY
        )

    def _make_static_loader(self, X, y, shuffle=False):
        """Static val loader. persistent_workers=True ONLY when NUM_WORKERS > 0."""
        pw = (self.config.NUM_WORKERS > 0)    # ← FIXED: was wrong hasattr check
        return DataLoader(
            TensorDataset(
                torch.tensor(X, dtype=torch.float32),
                torch.tensor(y, dtype=torch.long)
            ),
            batch_size         = self.config.BATCH_SIZE,
            shuffle            = shuffle,
            num_workers        = self.config.NUM_WORKERS,
            pin_memory         = self.config.PIN_MEMORY,
            persistent_workers = pw,
        )

    def _make_train_loader(self, X, y):
        """Dynamic train loader — num_workers=0 (no spawn overhead)."""
        return DataLoader(
            TensorDataset(
                torch.tensor(X, dtype=torch.float32),
                torch.tensor(y, dtype=torch.long)
            ),
            batch_size  = self.config.BATCH_SIZE,
            shuffle     = True,
            num_workers = 0,
            pin_memory  = False,
        )

    def set_weights(self, weights: Dict[str, torch.Tensor]):
        """Load global weights AND reset optimizer (fresh Adam moments)."""
        self.model.load_state_dict(weights, strict=True)
        self.model.to(self.device)
        self._reset_optimizer()

    def get_weights(self) -> Dict[str, torch.Tensor]:
        return {k: v.detach().cpu().clone() for k, v in self.model.state_dict().items()}

    def train_local(self, y_train_used: np.ndarray,
                    epochs: int, round_num: int) -> Tuple[float, float, float]:
        """Train on (possibly attacked) labels. Returns (train_loss, train_acc, val_loss)."""
        train_loader = self._make_train_loader(self.X_train_orig, y_train_used)
        self.model.train()
        all_losses, all_accs = [], []

        for _ in range(epochs):
            for Xb, yb in train_loader:
                Xb = Xb.to(self.device, non_blocking=True)
                yb = yb.to(self.device, non_blocking=True)
                self.optimizer.zero_grad()
                out  = self.model(Xb)
                loss = self.criterion(out, yb)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(
                    self.model.parameters(), self.config.MAX_GRAD_NORM)
                self.optimizer.step()
                all_losses.append(loss.item())
                all_accs.append((out.argmax(1) == yb).float().mean().item())

        val_loss = self._evaluate_val()
        return float(np.mean(all_losses)), float(np.mean(all_accs)), val_loss

    def _evaluate_val(self) -> float:
        self.model.eval()
        losses = []
        with torch.no_grad():
            for Xb, yb in self.val_loader:
                Xb = Xb.to(self.device, non_blocking=True)
                yb = yb.to(self.device, non_blocking=True)
                losses.append(self.criterion(self.model(Xb), yb).item())
        return float(np.mean(losses))


class FederatedServer:

    def __init__(self, model, X_test: np.ndarray, y_test: np.ndarray, config):
        self.config    = config
        self.device    = config.DEVICE
        self.model     = model.to(self.device)
        self.criterion = nn.CrossEntropyLoss()
        self.y_test    = y_test.copy()   # exposed for ASR computation in S10

        pw = (config.NUM_WORKERS > 0)    # ← FIXED: was wrong hasattr check
        self.test_loader = DataLoader(
            TensorDataset(
                torch.tensor(X_test, dtype=torch.float32),
                torch.tensor(y_test, dtype=torch.long)
            ),
            batch_size         = config.BATCH_SIZE * 2,
            shuffle            = False,
            num_workers        = config.NUM_WORKERS,
            pin_memory         = config.PIN_MEMORY,
            persistent_workers = pw,
        )
        print(f"  ✓ Server | test={len(y_test):,} | device={self.device}")

    def get_weights(self) -> Dict[str, torch.Tensor]:
        return {k: v.detach().cpu().clone() for k, v in self.model.state_dict().items()}

    def set_weights(self, weights: Dict[str, torch.Tensor]):
        self.model.load_state_dict(weights, strict=True)
        self.model.to(self.device)

    def evaluate(self) -> Dict:
        """Full evaluation with confusion matrix, FPR, FNR."""
        self.model.eval()
        preds, labels = [], []
        with torch.no_grad():
            for Xb, yb in self.test_loader:
                Xb = Xb.to(self.device, non_blocking=True)
                yb = yb.to(self.device, non_blocking=True)
                preds.extend(self.model(Xb).argmax(1).cpu().numpy().tolist())
                labels.extend(yb.cpu().numpy().tolist())

        preds  = np.array(preds)
        labels = np.array(labels)

        cm = confusion_matrix(labels, preds, labels=[0, 1])
        tn, fp, fn, tp = (int(cm[0,0]), int(cm[0,1]),
                          int(cm[1,0]), int(cm[1,1])) if cm.shape == (2,2) else (0,0,0,0)

        return {
            'accuracy' : float(accuracy_score(labels, preds)),
            'precision': float(precision_score(labels, preds, average='binary', zero_division=0)),
            'recall'   : float(recall_score   (labels, preds, average='binary', zero_division=0)),
            'f1'       : float(f1_score       (labels, preds, average='binary', zero_division=0)),
            'tn': tn, 'fp': fp, 'fn': fn, 'tp': tp,
            'fpr': float(fp / (fp + tn)) if (fp + tn) > 0 else 0.0,
            'fnr': float(fn / (fn + tp)) if (fn + tp) > 0 else 0.0,
        }

    def get_predictions(self) -> np.ndarray:
        """Return predictions array — used by S10 to compute ASR."""
        self.model.eval()
        preds = []
        with torch.no_grad():
            for Xb, _ in self.test_loader:
                preds.extend(self.model(Xb.to(self.device, non_blocking=True)
                                        ).argmax(1).cpu().numpy().tolist())
        return np.array(preds)


# ── Verification ──────────────────────────────────────────────────────────────
print("="*60)
print("SECTION 6 VERIFICATION")
print("="*60)

if 'preprocessed_data' in globals() and 'global_test_preprocessed' in globals():
    _cid = 0
    _Xtr, _ytr = preprocessed_data[_cid]['train']
    _Xva, _yva = preprocessed_data[_cid]['val']
    _Xte, _yte = global_test_preprocessed

    _m   = create_model(GlobalConfig, seed=42)
    _cli = FederatedClient(_cid, _m, _Xtr[:500], _ytr[:500],
                           _Xva[:100], _yva[:100], GlobalConfig)
    _srv = FederatedServer(create_model(GlobalConfig, seed=42),
                           _Xte[:500], _yte[:500], GlobalConfig)

    _l, _a, _vl = _cli.train_local(_ytr[:500], epochs=1, round_num=0)
    print(f"  train_local : loss={_l:.4f} acc={_a:.4f} val_loss={_vl:.4f} ✓")
    _met = _srv.evaluate()
    print(f"  evaluate    : acc={_met['accuracy']:.4f} f1={_met['f1']:.4f} ✓")
    _pred = _srv.get_predictions()
    assert len(_pred) == 500, f"Expected 500, got {len(_pred)}"
    print(f"  get_preds   : shape={_pred.shape} ✓")
    del _cli, _srv, _m
    torch.cuda.empty_cache(); gc.collect()
    print("✓ Section 6 verified")
else:
    print("⚠ preprocessed_data not yet available (normal — runs before Section 4)")
    print("  Verification will pass after Section 4 has run.")

print("="*60)
print("✓ Section 6 ready — persistent_workers bug fixed")
print("="*60)

In [ ]:
"""
================================================================================
SECTION 7: BYZANTINE ATTACK SIMULATION (UNBIASED, REPRODUCIBLE)
================================================================================
IMPROVEMENTS FROM PREVIOUS VERSION:
- Random selection of malicious clients (not always first K)
- Seed-based reproducibility
- Proper label-flipping attack
- Attack logging and tracking
================================================================================
"""

import numpy as np
from typing import List, Dict

class ByzantineClientManager:
    """
    Manages Byzantine (malicious) client selection and attacks.
    
    Key Features:
    - Random malicious client selection (seed-based for reproducibility)
    - Label-flipping attack (0→1, 1→0)
    - Attack tracking per client per round
    """
    
    def __init__(self, n_clients: int, malicious_ratio: float, seed: int = 42):
        """
        Initialize Byzantine attack manager.
        
        Args:
            n_clients: Total number of clients
            malicious_ratio: Fraction of malicious clients (0.0 to 1.0)
            seed: Random seed for reproducible malicious selection
        """
        self.n_clients = n_clients
        self.malicious_ratio = malicious_ratio
        self.seed = seed
        
        # Select malicious clients RANDOMLY using seed
        if malicious_ratio > 0:
            n_malicious = max(1, int(n_clients * malicious_ratio))
            rng = np.random.RandomState(seed)
            self.malicious_clients = sorted(
                rng.choice(n_clients, n_malicious, replace=False).tolist()
            )
        else:
            self.malicious_clients = []
        
        self.benign_clients = [
            i for i in range(n_clients) 
            if i not in self.malicious_clients
        ]
        
        # Attack tracking
        self.attack_counts = {i: 0 for i in self.malicious_clients}
        
        # Report
        print(f"\n{'='*70}")
        print(f"BYZANTINE ATTACK CONFIGURATION")
        print(f"{'='*70}")
        print(f"Total clients: {n_clients}")
        print(f"Malicious ratio: {malicious_ratio:.0%}")
        print(f"Malicious clients: {len(self.malicious_clients)} → {self.malicious_clients}")
        print(f"Benign clients: {len(self.benign_clients)} → {self.benign_clients}")
        print(f"Random seed: {seed}")
        print(f"{'='*70}")
    
    def is_malicious(self, client_id: int) -> bool:
        """Check if client is malicious."""
        return client_id in self.malicious_clients
    
    def apply_attack_to_client_data(
        self, 
        client_id: int, 
        y_train_clean: np.ndarray,
        round_num: int
    ) -> np.ndarray:
        """
        Apply label-flipping attack to client's training labels.
        
        Label-flipping attack:
        - Benign (0) → Attack (1)
        - Attack (1) → Benign (0)
        
        Args:
            client_id: Client ID
            y_train_clean: Original clean labels
            round_num: Current FL round
        
        Returns:
            y_train_attacked: Poisoned labels (if malicious) or clean labels (if benign)
        """
        if not self.is_malicious(client_id):
            # Benign client → return clean labels
            return y_train_clean.copy()
        
        # Malicious client → flip ALL labels
        y_train_attacked = 1 - y_train_clean  # 0→1, 1→0
        
        # Track attack
        self.attack_counts[client_id] += 1
        
        return y_train_attacked
    
    def get_attack_summary(self) -> Dict:
        """Get summary of attacks performed."""
        return {
            'n_malicious': len(self.malicious_clients),
            'malicious_clients': self.malicious_clients,
            'benign_clients': self.benign_clients,
            'attack_counts': self.attack_counts.copy()
        }


def compute_attack_success_rate(
    y_true: np.ndarray,
    y_pred: np.ndarray
) -> float:
    """
    Compute Attack Success Rate (ASR).
    
    ASR measures the percentage of attack samples misclassified as benign.
    This indicates how well the attack evaded detection.
    
    ASR = (False Negatives) / (Total Attack Samples) × 100
    
    Args:
        y_true: True labels
        y_pred: Predicted labels
    
    Returns:
        ASR percentage (0-100)
    """
    # Get attack samples (label = 1)
    attack_mask = (y_true == 1)
    
    if attack_mask.sum() == 0:
        return 0.0  # No attack samples
    
    # Count attacks misclassified as benign
    false_negatives = ((y_true == 1) & (y_pred == 0)).sum()
    total_attacks = attack_mask.sum()
    
    asr = (false_negatives / total_attacks) * 100.0
    
    return float(asr)


def compute_accuracy_drop(
    accuracy_baseline: float,
    accuracy_under_attack: float
) -> float:
    """
    Compute accuracy drop due to Byzantine attack.
    
    Accuracy Drop = Baseline Accuracy - Attack Accuracy
    
    Positive value → attack degraded performance
    Negative value → accuracy improved (suspicious, possible issue)
    
    Args:
        accuracy_baseline: Accuracy with 0% malicious clients
        accuracy_under_attack: Accuracy under attack
    
    Returns:
        Accuracy drop (absolute difference)
    """
    return float(accuracy_baseline - accuracy_under_attack)


# =============================================================================
# VERIFICATION
# =============================================================================

print("="*80)
print("BYZANTINE ATTACK VERIFICATION")
print("="*80)

# Test 1: Manager creation with different ratios
print("\nTest 1: Creating Byzantine managers...")
for ratio in [0.0, 0.2, 0.4]:
    manager = ByzantineClientManager(n_clients=9, malicious_ratio=ratio, seed=42)
    assert len(manager.malicious_clients) == max(0, int(9 * ratio))
    print(f"✓ {ratio:.0%} malicious: {manager.malicious_clients}")

# Test 2: Label flipping
print("\nTest 2: Testing label-flipping attack...")
manager = ByzantineClientManager(n_clients=9, malicious_ratio=0.2, seed=42)
y_clean = np.array([0, 0, 1, 1, 0, 1])

for client_id in range(9):
    y_attacked = manager.apply_attack_to_client_data(client_id, y_clean, round_num=0)
    
    if manager.is_malicious(client_id):
        assert not np.array_equal(y_clean, y_attacked), f"Client {client_id} should have flipped labels"
        assert np.array_equal(y_attacked, 1 - y_clean), f"Client {client_id} labels not properly flipped"
        print(f"✓ Client {client_id} (malicious): labels flipped correctly")
    else:
        assert np.array_equal(y_clean, y_attacked), f"Client {client_id} should have clean labels"
        print(f"✓ Client {client_id} (benign): labels unchanged")

# Test 3: ASR computation
print("\nTest 3: Testing ASR computation...")
y_true = np.array([0, 0, 1, 1, 1, 1])
y_pred_good = np.array([0, 0, 1, 1, 1, 1])  # Perfect → ASR = 0%
y_pred_bad = np.array([0, 0, 0, 0, 0, 0])   # All attacks missed → ASR = 100%

asr_good = compute_attack_success_rate(y_true, y_pred_good)
asr_bad = compute_attack_success_rate(y_true, y_pred_bad)

print(f"✓ Perfect predictions: ASR = {asr_good:.1f}% (expected 0%)")
print(f"✓ All attacks missed: ASR = {asr_bad:.1f}% (expected 100%)")

assert asr_good == 0.0
assert asr_bad == 100.0

# Test 4: Accuracy drop
print("\nTest 4: Testing accuracy drop...")
acc_drop = compute_accuracy_drop(accuracy_baseline=0.95, accuracy_under_attack=0.60)
print(f"✓ Accuracy drop: {acc_drop:.2f} (expected 0.35)")
assert abs(acc_drop - 0.35) < 1e-6

print("\n" + "="*80)
print("✓ All Byzantine attack tests passed")
print("="*80)


In [ ]:
"""
================================================================================
SECTION 8: TRUST MECHANISM (REALISTIC HONEYPOT SIMULATION)
================================================================================
IMPROVEMENTS:
- Realistic detection rate (90% vs previous 85%)
- Lower false positive rate (8% vs previous 10%)
- Aggressive trust decay for detected malicious clients
- Probabilistic detection per round
- Seed-based reproducibility
================================================================================
"""

import numpy as np
from typing import List, Dict

class TrustScoreManager:
    """
    Simulates honeypot-based trust scoring for federated clients.
    
    Trust Mechanism:
    - Each FL round, clients may be detected by honeypot
    - Malicious clients: 90% chance of detection per round
    - Benign clients: 8% chance of false positive per round
    - Detected clients have their trust score reduced
    - Trust scores weight client contributions in aggregation
    """
    
    def __init__(
        self, 
        n_clients: int, 
        malicious_clients: List[int],
        config: GlobalConfig,
        seed: int = 42
    ):
        """
        Initialize Trust Score Manager.
        
        Args:
            n_clients: Total number of clients
            malicious_clients: List of malicious client IDs
            config: GlobalConfig with trust parameters
            seed: Random seed for reproducibility
        """
        self.n_clients = n_clients
        self.malicious_clients = set(malicious_clients)
        self.benign_clients = set(range(n_clients)) - self.malicious_clients
        self.seed = seed
        self.config = config
        
        # Trust scores (all start at 1.0)
        self.trust_scores = {i: config.INITIAL_TRUST for i in range(n_clients)}
        
        # Detection tracking
        self.detection_counts = {i: 0 for i in range(n_clients)}
        self.detection_history = {i: [] for i in range(n_clients)}
        
        # Honeypot parameters
        self.detection_rate = config.MALICIOUS_DETECTION_RATE  # 90%
        self.false_positive_rate = config.BENIGN_FPR  # 8%
        self.trust_decay = config.TRUST_DECAY_AGGRESSIVE  # {0: 1.0, 1: 0.5, 2: 0.25, 3: 0.15}
        self.min_trust = config.MIN_TRUST  # 0.15
        
        print(f"\n{'='*70}")
        print(f"TRUST MECHANISM CONFIGURATION")
        print(f"{'='*70}")
        print(f"Total clients: {n_clients}")
        print(f"Malicious clients: {len(self.malicious_clients)} → {sorted(self.malicious_clients)}")
        print(f"Benign clients: {len(self.benign_clients)} → {sorted(self.benign_clients)}")
        print(f"Malicious detection rate: {self.detection_rate:.0%}")
        print(f"Benign false positive rate: {self.false_positive_rate:.0%}")
        print(f"Trust decay schedule: {self.trust_decay}")
        print(f"Minimum trust: {self.min_trust}")
        print(f"Random seed: {seed}")
        print(f"{'='*70}")
    
    def update_trust_scores(self, round_num: int):
        """
        Update trust scores based on simulated honeypot detections.
        
        Each round:
        1. Roll dice for each client
        2. Malicious: detected with 90% probability
        3. Benign: false positive with 8% probability
        4. Update trust scores based on detection history
        
        Args:
            round_num: Current FL round
        """
        # Create round-specific RNG for reproducibility
        rng = np.random.RandomState(self.seed + round_num)
        
        detections_this_round = []
        
        for client_id in range(self.n_clients):
            is_malicious = client_id in self.malicious_clients
            
            # Determine if detected this round
            if is_malicious:
                # Malicious client: detect with detection_rate probability
                detected = rng.rand() < self.detection_rate
            else:
                # Benign client: false positive with false_positive_rate
                detected = rng.rand() < self.false_positive_rate
            
            if detected:
                self.detection_counts[client_id] += 1
                self.detection_history[client_id].append(round_num)
                detections_this_round.append((client_id, is_malicious))
                
                # Update trust score based on detection count
                detection_count = self.detection_counts[client_id]
                
                if detection_count in self.trust_decay:
                    new_trust = self.trust_decay[detection_count]
                else:
                    # If more detections than in schedule, use min_trust
                    new_trust = self.min_trust
                
                self.trust_scores[client_id] = max(new_trust, self.min_trust)
        
        # Log detections
        if detections_this_round:
            print(f"\nRound {round_num} Honeypot Detections:")
            for client_id, is_mal in detections_this_round:
                tag = "MALICIOUS" if is_mal else "BENIGN-FP"
                count = self.detection_counts[client_id]
                trust = self.trust_scores[client_id]
                print(f"  Client {client_id} ({tag}): detection #{count} → trust = {trust:.3f}")
    
    def get_trust_score(self, client_id: int) -> float:
        """Get current trust score for a client."""
        return self.trust_scores[client_id]
    
    def get_trust_weights(self) -> Dict[int, float]:
        """Get trust scores for all clients as weights."""
        return self.trust_scores.copy()
    
    def get_trust_summary(self) -> Dict:
        """Get summary of trust scores and detections."""
        malicious_trusts = [self.trust_scores[c] for c in self.malicious_clients]
        benign_trusts = [self.trust_scores[c] for c in self.benign_clients]
        
        malicious_detected = sum(1 for c in self.malicious_clients if self.detection_counts[c] > 0)
        benign_fp = sum(1 for c in self.benign_clients if self.detection_counts[c] > 0)
        
        return {
            'trust_scores': self.trust_scores.copy(),
            'detection_counts': self.detection_counts.copy(),
            'avg_trust_malicious': float(np.mean(malicious_trusts)) if malicious_trusts else 1.0,
            'avg_trust_benign': float(np.mean(benign_trusts)) if benign_trusts else 1.0,
            'malicious_detection_rate': float(malicious_detected / len(self.malicious_clients)) if self.malicious_clients else 0.0,
            'benign_fp_rate': float(benign_fp / len(self.benign_clients)) if self.benign_clients else 0.0
        }


# =============================================================================
# VERIFICATION
# =============================================================================

print("="*80)
print("TRUST MECHANISM VERIFICATION")
print("="*80)

# Test 1: Manager creation
print("\nTest 1: Creating Trust Manager...")
test_malicious = [0, 1, 3]  # 3 malicious clients
test_manager = TrustScoreManager(
    n_clients=9,
    malicious_clients=test_malicious,
    config=GlobalConfig,
    seed=42
)
print("✓ Trust Manager created")

# Test 2: Initial trust scores
print("\nTest 2: Verifying initial trust scores...")
for client_id in range(9):
    trust = test_manager.get_trust_score(client_id)
    assert trust == 1.0, f"Client {client_id} should start with trust=1.0"
print(f"✓ All clients initialized with trust = 1.0")

# Test 3: Simulate detections over multiple rounds
print("\nTest 3: Simulating detections over 10 rounds...")
for round_num in range(10):
    test_manager.update_trust_scores(round_num)

# Test 4: Verify trust scores changed
print("\nTest 4: Verifying trust score changes...")
trust_summary = test_manager.get_trust_summary()
print(f"\nTrust Summary after 10 rounds:")
print(f"  Malicious clients avg trust: {trust_summary['avg_trust_malicious']:.3f}")
print(f"  Benign clients avg trust: {trust_summary['avg_trust_benign']:.3f}")
print(f"  Malicious detection rate: {trust_summary['malicious_detection_rate']:.1%}")
print(f"  Benign false positive rate: {trust_summary['benign_fp_rate']:.1%}")

# Verify malicious clients have lower trust (on average)
assert trust_summary['avg_trust_malicious'] < trust_summary['avg_trust_benign'], \
    "Malicious clients should have lower average trust"

print(f"\n✓ Malicious clients detected and penalized")
print(f"✓ Benign clients maintain higher trust")

print("\n" + "="*80)
print("✓ All trust mechanism tests passed")
print("="*80)


In [ ]:
"""
================================================================================
SECTION 9: AGGREGATION METHODS (DTYPE-SAFE, NUMPY-BASED)
================================================================================
Methods Implemented:
1. FedAvg: Simple averaging (no defense)
2. Krum: Distance-based robust aggregation
3. Geometric Median (GM): Coordinate-wise median
4. Trust-Weighted GM: GM weighted by trust scores (OUR CONTRIBUTION)

All methods work in float64 numpy space to avoid dtype issues.
================================================================================
"""

import numpy as np
import torch
from typing import List, Dict, Optional
import copy

def state_dict_to_vector(state_dict: Dict[str, torch.Tensor]) -> np.ndarray:
    """
    Convert PyTorch state_dict to flat numpy vector.
    
    Returns:
        vector: 1D numpy array (float64)
    """
    vectors = []
    for key in sorted(state_dict.keys()):
        tensor = state_dict[key]
        vectors.append(tensor.detach().cpu().numpy().ravel().astype(np.float64))
    
    return np.concatenate(vectors)


def vector_to_state_dict(
    vector: np.ndarray,
    template_state_dict: Dict[str, torch.Tensor]
) -> Dict[str, torch.Tensor]:
    """
    Convert flat numpy vector back to PyTorch state_dict.
    
    Args:
        vector: 1D numpy array
        template_state_dict: Template to get shapes and dtypes
    
    Returns:
        state_dict with same structure as template
    """
    state_dict = {}
    idx = 0
    
    for key in sorted(template_state_dict.keys()):
        tensor = template_state_dict[key]
        shape = tensor.shape
        dtype = tensor.dtype
        size = tensor.numel()
        
        # Extract slice from vector
        param_vector = vector[idx:idx+size]
        
        # Reshape and convert to original dtype
        param_tensor = torch.from_numpy(param_vector).reshape(shape).to(dtype)
        
        state_dict[key] = param_tensor
        idx += size
    
    return state_dict


def fedavg_aggregate(
    client_weights: List[Dict[str, torch.Tensor]],
    client_sizes: Optional[List[int]] = None
) -> Dict[str, torch.Tensor]:
    """
    FedAvg: Weighted average aggregation.
    
    Args:
        client_weights: List of client model weights
        client_sizes: Optional list of client dataset sizes (for weighted avg)
    
    Returns:
        Aggregated model weights
    """
    if client_sizes is None:
        # Uniform weighting
        client_sizes = [1.0] * len(client_weights)
    
    total_size = sum(client_sizes)
    weights_norm = [size / total_size for size in client_sizes]
    
    # Convert to vectors
    vectors = [state_dict_to_vector(w) for w in client_weights]
    
    # Weighted average
    avg_vector = np.zeros_like(vectors[0])
    for vec, weight in zip(vectors, weights_norm):
        avg_vector += vec * weight
    
    # Convert back to state_dict
    aggregated = vector_to_state_dict(avg_vector, client_weights[0])
    
    return aggregated


def krum_aggregate(
    client_weights: List[Dict[str, torch.Tensor]],
    n_malicious: int = 0
) -> Dict[str, torch.Tensor]:
    """
    Krum: Select one client with smallest distance sum to nearest neighbors.
    
    Krum is robust to up to f Byzantine clients (n_malicious).
    
    Args:
        client_weights: List of client model weights
        n_malicious: Number of malicious clients to tolerate
    
    Returns:
        Selected client's weights
    """
    n_clients = len(client_weights)
    
    # Krum parameter: number of closest neighbors to consider
    n_closest = n_clients - n_malicious - 2
    
    if n_closest < 1:
        # Not enough clients for Krum, fallback to first client
        print(f"⚠ Krum: n_closest={n_closest} < 1, using first client")
        return client_weights[0]
    
    # Convert to vectors
    vectors = [state_dict_to_vector(w) for w in client_weights]
    vectors = np.array(vectors)  # Shape: (n_clients, param_dim)
    
    # Compute pairwise distances
    distances = np.zeros((n_clients, n_clients))
    for i in range(n_clients):
        for j in range(i+1, n_clients):
            dist = np.linalg.norm(vectors[i] - vectors[j])
            distances[i, j] = dist
            distances[j, i] = dist
    
    # For each client, compute score = sum of distances to n_closest neighbors
    scores = []
    for i in range(n_clients):
        # Get distances to all other clients
        dists_to_others = distances[i].copy()
        dists_to_others[i] = np.inf  # Exclude self
        
        # Get n_closest smallest distances
        closest_dists = np.partition(dists_to_others, n_closest)[:n_closest]
        score = closest_dists.sum()
        scores.append(score)
    
    # Select client with minimum score
    selected_idx = np.argmin(scores)
    
    return client_weights[selected_idx]


def geometric_median_aggregate(
    client_weights: List[Dict[str, torch.Tensor]],
    max_iters: int = 100,
    tol: float = 1e-5
) -> Dict[str, torch.Tensor]:
    """
    Geometric Median (Weiszfeld's algorithm): Coordinate-wise median.
    
    GM is robust to Byzantine attacks by finding the point that minimizes
    sum of distances to all clients.
    
    Args:
        client_weights: List of client model weights
        max_iters: Maximum iterations for Weiszfeld
        tol: Convergence tolerance
    
    Returns:
        Aggregated model weights (geometric median)
    """
    # Convert to vectors
    vectors = [state_dict_to_vector(w) for w in client_weights]
    vectors = np.array(vectors)  # Shape: (n_clients, param_dim)
    
    # Initialize with component-wise median (good starting point)
    median = np.median(vectors, axis=0)
    
    # Weiszfeld's algorithm
    for iteration in range(max_iters):
        prev_median = median.copy()
        
        # Compute distances from median to each client
        distances = np.linalg.norm(vectors - median, axis=1)
        
        # Avoid division by zero
        distances = np.maximum(distances, 1e-10)
        
        # Weighted update
        weights = 1.0 / distances
        weights /= weights.sum()
        
        # Update median
        median = np.sum(vectors * weights[:, np.newaxis], axis=0)
        
        # Check convergence
        change = np.linalg.norm(median - prev_median)
        if change < tol:
            break
    
    # Convert back to state_dict
    aggregated = vector_to_state_dict(median, client_weights[0])
    
    return aggregated


def trust_weighted_gm_aggregate(
    client_weights: List[Dict[str, torch.Tensor]],
    trust_scores: Dict[int, float],
    client_ids: List[int],
    max_iters: int = 100,
    tol: float = 1e-5
) -> Dict[str, torch.Tensor]:
    """
    Trust-Weighted Geometric Median: GM with trust-based weighting.
    
    THIS IS OUR MAIN CONTRIBUTION.
    
    Combines:
    - Geometric median (Byzantine-robust)
    - Trust scores from honeypot (behavioral defense)
    
    Clients with low trust have less influence on the global model.
    
    Args:
        client_weights: List of client model weights
        trust_scores: Dict mapping client_id to trust score [0, 1]
        client_ids: List of client IDs (same order as client_weights)
        max_iters: Maximum iterations
        tol: Convergence tolerance
    
    Returns:
        Aggregated model weights
    """
    # Convert to vectors
    vectors = [state_dict_to_vector(w) for w in client_weights]
    vectors = np.array(vectors)  # Shape: (n_clients, param_dim)
    
    # Get trust scores (normalized)
    trust_values = np.array([trust_scores[cid] for cid in client_ids])
    
    # Normalize trust scores
    if trust_values.sum() > 0:
        trust_normalized = trust_values / trust_values.sum()
    else:
        # Fallback to uniform if all zero
        trust_normalized = np.ones_like(trust_values) / len(trust_values)
    
    # Initialize with trust-weighted average (better starting point than median)
    median = np.sum(vectors * trust_normalized[:, np.newaxis], axis=0)
    
    # Weiszfeld's algorithm with trust weighting
    for iteration in range(max_iters):
        prev_median = median.copy()
        
        # Compute distances
        distances = np.linalg.norm(vectors - median, axis=1)
        distances = np.maximum(distances, 1e-10)  # Avoid division by zero
        
        # Combine distance-based weights with trust
        distance_weights = 1.0 / distances
        combined_weights = distance_weights * trust_values
        
        # Normalize
        if combined_weights.sum() > 0:
            combined_weights /= combined_weights.sum()
        else:
            combined_weights = trust_normalized
        
        # Update median
        median = np.sum(vectors * combined_weights[:, np.newaxis], axis=0)
        
        # Check convergence
        change = np.linalg.norm(median - prev_median)
        if change < tol:
            break
    
    # Convert back to state_dict
    aggregated = vector_to_state_dict(median, client_weights[0])
    
    return aggregated


def aggregate_models(
    method: str,
    client_weights: List[Dict[str, torch.Tensor]],
    client_ids: List[int],
    client_sizes: Optional[List[int]] = None,
    trust_scores: Optional[Dict[int, float]] = None,
    n_malicious: int = 0,
    config: GlobalConfig = None
) -> Dict[str, torch.Tensor]:
    """
    Universal aggregation function.
    
    Args:
        method: "fedavg", "krum", "gm", or "trust_gm"
        client_weights: List of client model state_dicts
        client_ids: List of client IDs
        client_sizes: Optional dataset sizes (for FedAvg)
        trust_scores: Optional trust scores (for Trust-GM)
        n_malicious: Number of malicious clients (for Krum)
        config: GlobalConfig (for GM/Trust-GM parameters)
    
    Returns:
        Aggregated model state_dict
    """
    if method == "fedavg":
        return fedavg_aggregate(client_weights, client_sizes)
    
    elif method == "krum":
        return krum_aggregate(client_weights, n_malicious)
    
    elif method == "gm":
        max_iters = config.GM_MAX_ITERS if config else 100
        tol = config.GM_TOL if config else 1e-5
        return geometric_median_aggregate(client_weights, max_iters, tol)
    
    elif method == "trust_gm":
        if trust_scores is None:
            raise ValueError("trust_gm requires trust_scores")
        max_iters = config.GM_MAX_ITERS if config else 100
        tol = config.GM_TOL if config else 1e-5
        return trust_weighted_gm_aggregate(
            client_weights, trust_scores, client_ids, max_iters, tol
        )
    
    else:
        raise ValueError(f"Unknown aggregation method: {method}")


# =============================================================================
# VERIFICATION
# =============================================================================

print("="*80)
print("AGGREGATION METHODS VERIFICATION")
print("="*80)

# Create dummy client models
print("\nCreating dummy client models...")
n_test_clients = 5
test_models = [create_model(GlobalConfig) for _ in range(n_test_clients)]
test_weights = [m.state_dict() for m in test_models]
test_client_ids = list(range(n_test_clients))

# Test 1: FedAvg
print("\nTest 1: FedAvg aggregation...")
fedavg_result = aggregate_models(
    method="fedavg",
    client_weights=test_weights,
    client_ids=test_client_ids
)
print(f"✓ FedAvg: {len(fedavg_result)} parameters aggregated")

# Test 2: Krum
print("\nTest 2: Krum aggregation...")
krum_result = aggregate_models(
    method="krum",
    client_weights=test_weights,
    client_ids=test_client_ids,
    n_malicious=1
)
print(f"✓ Krum: {len(krum_result)} parameters aggregated")

# Test 3: Geometric Median
print("\nTest 3: Geometric Median aggregation...")
gm_result = aggregate_models(
    method="gm",
    client_weights=test_weights,
    client_ids=test_client_ids,
    config=GlobalConfig
)
print(f"✓ GM: {len(gm_result)} parameters aggregated")

# Test 4: Trust-Weighted GM
print("\nTest 4: Trust-Weighted GM aggregation...")
test_trust_scores = {i: 1.0 if i < 3 else 0.3 for i in range(n_test_clients)}
trust_gm_result = aggregate_models(
    method="trust_gm",
    client_weights=test_weights,
    client_ids=test_client_ids,
    trust_scores=test_trust_scores,
    config=GlobalConfig
)
print(f"✓ Trust-GM: {len(trust_gm_result)} parameters aggregated")
print(f"  Trust scores used: {test_trust_scores}")

# Verify all methods return valid state_dicts
for method, result in [("FedAvg", fedavg_result), ("Krum", krum_result), 
                        ("GM", gm_result), ("Trust-GM", trust_gm_result)]:
    assert isinstance(result, dict), f"{method} should return dict"
    assert len(result) == len(test_weights[0]), f"{method} should have same keys as input"
    print(f"✓ {method} result structure valid")

# Cleanup
del test_models, test_weights
torch.cuda.empty_cache()

print("\n" + "="*80)
print("✓ All aggregation methods verified successfully")
print("="*80)


In [ ]:
"""
================================================================================
SECTION 10: MAIN TRAINING LOOP & EXPERIMENTAL PROTOCOL
================================================================================
FIXES vs UPLOADED VERSION:
1. SEED BUG: create_model now receives experiment seed → unique model init
2. TRAIN_LOCAL API: called as client.train_local(y_used, epochs, round_num)
3. SERVER.TEST_LOADER: ASR uses server.get_predictions() + server.y_test
4. N_SEEDS=1: single clean seed, honest for mentor (multi-seed on college PCs)
5. TIMING: FL_ROUNDS=25 + BATCH=512 + LOCAL_EPOCHS=2 → ~14s/round (~2h total)
6. ORDERING: prioritised (worst case first) — kept from uploaded code ✓
================================================================================
"""

import time, json, gc, copy
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Tuple

import numpy as np
import torch
import pandas as pd


def set_seed(seed: int):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False


def save_checkpoint(exp_id: str, round_history: List, final_metrics: Dict, config):
    cp = {
        'experiment_id': exp_id,
        'round_history': round_history,
        'final_metrics': final_metrics,
        'timestamp': datetime.now().isoformat()
    }
    path = config.CHECKPOINT_DIR / f"{exp_id}.json"
    with open(path, 'w') as f:
        json.dump(cp, f, indent=2)
    print(f"  ✓ Checkpoint: {path.name}")


def run_single_experiment(
    exp_id: str,
    method: str,
    malicious_ratio: float,
    seed: int,
    preprocessed_data: Dict,
    global_test_data: Tuple,
    config
) -> Tuple[List[Dict], Dict]:

    print(f"\n{'='*58}")
    print(f"  {exp_id}")
    print(f"  Method={method.upper()} | Attack={malicious_ratio:.0%} | Seed={seed}")
    print(f"{'='*58}")

    # ── Set all seeds ──────────────────────────────────────────────────
    set_seed(seed)

    X_test, y_test = global_test_data

    # ── Byzantine manager ─────────────────────────────────────────────
    byz_mgr = ByzantineClientManager(
        n_clients=config.N_CLIENTS,
        malicious_ratio=malicious_ratio,
        seed=seed
    )

    # ── Trust manager (only initialised for trust_gm) ─────────────────
    trust_mgr = None
    if method == "trust_gm":
        trust_mgr = TrustScoreManager(
            n_clients=config.N_CLIENTS,
            malicious_clients=byz_mgr.malicious_clients,
            config=config,
            seed=seed
        )

    # ── Global model — SEED FIX: use experiment seed ──────────────────
    global_model = create_model(config, seed=seed)   # ← FIX

    # ── Server ────────────────────────────────────────────────────────
    server = FederatedServer(
        model=global_model,
        X_test=X_test,
        y_test=y_test,
        config=config
    )

    # ── Clients ───────────────────────────────────────────────────────
    clients, client_sizes, client_ids = [], [], []
    for cid in range(config.N_CLIENTS):
        X_tr, y_tr = preprocessed_data[cid]['train']
        X_va, y_va = preprocessed_data[cid]['val']
        c_model = create_model(config, seed=seed + cid)   # unique per client
        c_model.load_state_dict(server.get_weights())
        client = FederatedClient(cid, c_model, X_tr, y_tr, X_va, y_va, config)
        clients.append(client)
        client_sizes.append(len(X_tr))
        client_ids.append(cid)

    # ── Aggregation config ─────────────────────────────────────────────
    agg_config        = copy.copy(config)
    agg_config.KRUM_F = len(byz_mgr.malicious_clients)

    round_history = []

    # ── Round 0 evaluation (initial random model) ─────────────────────
    m0 = server.evaluate()
    round_history.append({'round': 0, **m0})
    print(f"  R00 | Acc={m0['accuracy']:.4f} F1={m0['f1']:.4f} "
          f"TP={m0['tp']:,} FP={m0['fp']:,} FN={m0['fn']:,} "
          f"FPR={m0['fpr']:.3f} FNR={m0['fnr']:.3f}")

    # ── FL Rounds ─────────────────────────────────────────────────────
    start = time.time()

    for rnd in range(1, config.FL_ROUNDS + 1):
        t0 = time.time()

        # Update trust scores (trust_gm only)
        if trust_mgr:
            trust_mgr.update_trust_scores(rnd - 1)

        # Broadcast global weights
        gw = server.get_weights()
        for client in clients:
            client.set_weights(gw)

        # Local training (with attacked labels for malicious clients)
        for client in clients:
            y_used = byz_mgr.apply_attack_to_client_data(
                client.client_id, client.y_train_clean, rnd)
            client.train_local(y_used, epochs=config.LOCAL_EPOCHS, round_num=rnd)

        # Collect weights
        w_list = [c.get_weights() for c in clients]

        # Get trust scores if needed
        trust_scores = trust_mgr.get_trust_weights() if trust_mgr else None

        # Aggregate
        agg_w = aggregate_models(
            method=method,
            client_weights=w_list,
            client_ids=client_ids,
            client_sizes=client_sizes,
            trust_scores=trust_scores,
            n_malicious=len(byz_mgr.malicious_clients),
            config=agg_config
        )
        server.set_weights(agg_w)

        # Evaluate
        m  = server.evaluate()
        rt = time.time() - t0

        rec = {'round': rnd, **m, 'time_sec': rt}
        if trust_mgr:
            ts = trust_mgr.get_trust_summary()
            rec['avg_trust_mal'] = ts['avg_trust_malicious']
            rec['avg_trust_ben'] = ts['avg_trust_benign']
        round_history.append(rec)

        if rnd % 5 == 0 or rnd == 1:
            elapsed = time.time() - start
            eta     = (elapsed / rnd) * (config.FL_ROUNDS - rnd)
            print(f"  R{rnd:02d} | Acc={m['accuracy']:.4f} F1={m['f1']:.4f} "
                  f"TP={m['tp']:,} FP={m['fp']:,} FN={m['fn']:,} "
                  f"FPR={m['fpr']:.3f} FNR={m['fnr']:.3f} "
                  f"[{rt:.1f}s | ETA {eta/60:.1f}m]")

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # ── Final metrics ─────────────────────────────────────────────────
    final = round_history[-1].copy()

    # ASR: fraction of attack samples misclassified as benign (FNR)
    # Uses server.get_predictions() and server.y_test (both exposed in S6 fix)
    y_pred = server.get_predictions()
    asr    = compute_attack_success_rate(server.y_test, y_pred)
    final['asr'] = asr

    if trust_mgr:
        ts = trust_mgr.get_trust_summary()
        final['avg_trust_malicious'] = ts['avg_trust_malicious']
        final['avg_trust_benign']    = ts['avg_trust_benign']

    final['training_time_sec'] = time.time() - start

    print(f"\n  ✓ DONE | Acc={final['accuracy']:.4f} F1={final['f1']:.4f} "
          f"ASR={asr:.2f}% | {final['training_time_sec']/60:.1f}min")

    del clients, server, global_model
    torch.cuda.empty_cache()
    gc.collect()

    return round_history, final


def run_full_experimental_protocol(
    preprocessed_data: Dict,
    global_test_data: Tuple,
    config=None
) -> pd.DataFrame:

    if config is None:
        config = GlobalConfig

    seed = config.SEED

    # ── Build experiment list (prioritised: critical first) ───────────
    experiments = []
    # P1: baselines (0%)
    for method in config.AGGREGATION_METHODS:
        experiments.append({'method': method, 'malicious': 0.0, 'priority': 1})
    # P2: worst case (40%)
    for method in config.AGGREGATION_METHODS:
        experiments.append({'method': method, 'malicious': 0.4, 'priority': 2})
    # P3: intermediate
    for ratio in [0.2, 0.3]:
        for method in config.AGGREGATION_METHODS:
            experiments.append({'method': method, 'malicious': ratio, 'priority': 3})

    n_exp     = len(experiments)
    est_min   = n_exp * config.FL_ROUNDS * 7 / 60
    print(f"\n{'#'*58}")
    print(f"  FULL EXPERIMENTAL PROTOCOL")
    print(f"{'#'*58}")
    print(f"  Methods  : {config.AGGREGATION_METHODS}")
    print(f"  Attacks  : {config.MALICIOUS_RATIOS}")
    print(f"  Rounds   : {config.FL_ROUNDS}")
    print(f"  Seed     : {seed}")
    print(f"  Total    : {n_exp} experiments")
    print(f"  Est time : ~{est_min:.0f} min (~{est_min/60:.1f}h)")
    print(f"  Ordering : baseline → 40% → 20% → 30% (critical first)")
    print(f"{'#'*58}\n")

    all_results   = []
    protocol_start = time.time()

    for idx, exp in enumerate(experiments):
        exp_id = f"E{idx:02d}_{exp['method']}_{int(exp['malicious']*100)}pct"

        print(f"\n{'#'*58}")
        print(f"  {idx+1}/{n_exp} | Priority {exp['priority']}")
        elapsed = time.time() - protocol_start
        if idx > 0:
            avg_per = elapsed / idx
            eta     = avg_per * (n_exp - idx)
            print(f"  Elapsed {elapsed/3600:.2f}h | ETA {eta/3600:.2f}h")
        print(f"{'#'*58}")

        hist, final = run_single_experiment(
            exp_id        = exp_id,
            method        = exp['method'],
            malicious_ratio = exp['malicious'],
            seed          = seed,
            preprocessed_data = preprocessed_data,
            global_test_data  = global_test_data,
            config        = config
        )

        save_checkpoint(exp_id, hist, final, config)

        all_results.append({
            'exp_id'  : exp_id,
            'method'  : exp['method'],
            'malicious': exp['malicious'],
            'seed'    : seed,
            **final
        })

        gc.collect()

    # ── Summary ───────────────────────────────────────────────────────
    df = pd.DataFrame(all_results)

    # Compute accuracy drop vs same-method baseline
    baseline_acc = {}
    for row in all_results:
        if row['malicious'] == 0.0:
            baseline_acc[row['method']] = row['accuracy']

    df['acc_drop'] = df.apply(
        lambda r: baseline_acc.get(r['method'], r['accuracy']) - r['accuracy'],
        axis=1
    )

    results_path  = config.RESULTS_DIR / "all_results.csv"
    df.to_csv(results_path, index=False)

    total_time = time.time() - protocol_start
    print(f"\n{'#'*58}")
    print(f"  ALL EXPERIMENTS COMPLETE")
    print(f"  Total time: {total_time/3600:.2f}h")
    print(f"  Results: {results_path}")
    print(f"{'#'*58}")

    # Print summary table
    cols = ['method', 'malicious', 'accuracy', 'f1', 'fpr', 'fnr', 'asr', 'acc_drop']
    cols = [c for c in cols if c in df.columns]
    print("\n" + df[cols].to_string(index=False))

    return df


# ── Ready ──────────────────────────────────────────────────────────────────
print("="*58)
print("  SECTION 10 READY")
print("="*58)

if 'preprocessed_data' not in globals():
    print("❌ preprocessed_data missing — run Sections 1-4 first")
elif 'global_test_preprocessed' not in globals():
    print("❌ global_test_preprocessed missing — run Section 4 first")
else:
    n_exp    = len(GlobalConfig.MALICIOUS_RATIOS) * len(GlobalConfig.AGGREGATION_METHODS)
    est_min  = n_exp * GlobalConfig.FL_ROUNDS * 7 / 60
    print(f"  ✓ preprocessed_data   ({len(preprocessed_data)} clients)")
    print(f"  ✓ global_test         ({len(global_test_preprocessed[0]):,} samples)")
    print(f"  Experiments: {n_exp} | Rounds: {GlobalConfig.FL_ROUNDS}")
    print(f"  Estimated  : ~{est_min:.0f} min")
    print()
    print("  Run: summary = run_full_experimental_protocol(")
    print("           preprocessed_data, global_test_preprocessed)")

In [ ]:
"""
================================================================================
SECTION 11: RESULTS VISUALIZATION & ANALYSIS (REWRITTEN)
================================================================================
Fixes vs current notebook:
1. Loads all_results.csv (not summary.csv which never exists)
2. Checkpoint glob fixed: uses re.compile — glob *_gm_Xpct.json wrongly
   matches trust_gm files (confirmed bug)
3. Stats test replaced: ttest_rel with n=1 crashes; now shows descriptive
   comparison with honest note about needing multiple seeds
4. Primary metric: FNR/ASR (not accuracy — misleading on imbalanced test set)
5. Added trust score convergence plot (per-round data from checkpoints)
================================================================================
"""

import re
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore")

# ── Style ─────────────────────────────────────────────────────────────────────
try:
    plt.style.use('seaborn-v0_8-darkgrid')
except OSError:
    plt.style.use('seaborn-darkgrid')   # older matplotlib fallback

METHODS       = ['fedavg', 'krum', 'gm', 'trust_gm']
METHOD_LABELS = {
    'fedavg'   : 'FedAvg',
    'krum'     : 'Krum',
    'gm'       : 'Geometric Median',
    'trust_gm' : 'Trust-GM (Ours)',
}
COLORS = {
    'fedavg'   : '#e74c3c',
    'krum'     : '#3498db',
    'gm'       : '#f39c12',
    'trust_gm' : '#27ae60',
}
LINESTYLES = {
    'fedavg'   : '--',
    'krum'     : '-.',
    'gm'       : ':',
    'trust_gm' : '-',
}


# ──────────────────────────────────────────────────────────────────────────────
# INTERNAL HELPERS
# ──────────────────────────────────────────────────────────────────────────────

def _find_checkpoint(config, method: str, malicious_pct: int):
    """
    Find checkpoint file for a (method, malicious_pct) pair.

    Uses re.compile NOT glob, because glob '*_gm_Xpct.json' incorrectly
    matches 'E03_trust_gm_0pct.json' as well as 'E02_gm_0pct.json'.

    Returns (Path, dict) or (None, None) if not found.
    """
    # Pattern: E{digits}_{method}_{pct}pct.json  — exact method match
    pattern = re.compile(
        r'^E\d+_' + re.escape(method) + r'_' + str(malicious_pct) + r'pct\.json$'
    )
    for p in config.CHECKPOINT_DIR.iterdir():
        if p.suffix == '.json' and pattern.match(p.name):
            with open(p) as f:
                return p, json.load(f)
    return None, None


def _load_results(config) -> pd.DataFrame:
    """Load all_results.csv. Returns None if missing."""
    path = config.RESULTS_DIR / "all_results.csv"
    if not path.exists():
        print(f"❌ {path} not found — run experiments first.")
        return None
    df = pd.read_csv(path)
    print(f"  ✓ Loaded {len(df)} experiment results from {path.name}")
    return df


# ──────────────────────────────────────────────────────────────────────────────
# PLOT 1: FNR (= ASR/100) vs FL Round  — PRIMARY metric
# ──────────────────────────────────────────────────────────────────────────────

def plot_fnr_vs_round(config, malicious_pct: int = 40):
    """
    FNR per round for all methods at a given attack level.
    FNR = fraction of attack traffic misclassified as benign = ASR/100.
    This is the PRIMARY metric: lower means the IDS misses fewer attacks.
    Data comes from per-round checkpoint JSON files.
    """
    fig, ax = plt.subplots(figsize=(12, 7))
    plotted = 0

    for method in METHODS:
        _, cp = _find_checkpoint(config, method, malicious_pct)
        if cp is None:
            print(f"  ⚠ No checkpoint: {method} {malicious_pct}%")
            continue

        hist = cp['round_history']
        rounds  = [r['round'] for r in hist]
        fnr_pct = [r['fnr'] * 100 for r in hist]   # FNR as % = ASR

        ax.plot(rounds, fnr_pct,
                label=METHOD_LABELS[method],
                color=COLORS[method],
                linestyle=LINESTYLES[method],
                linewidth=2.5, marker='o', markersize=4, markevery=5)
        plotted += 1

    if plotted == 0:
        print("  ⚠ No checkpoint data — run experiments first.")
        plt.close()
        return

    ax.set_xlabel('FL Round', fontsize=13)
    ax.set_ylabel('FNR (%) — Missed Attacks / Total Attacks', fontsize=13)
    ax.set_title(
        f'Attack Success Rate (FNR) vs FL Round — {malicious_pct}% Byzantine Clients\n'
        f'Lower = IDS misses fewer attacks = better Byzantine robustness',
        fontsize=13
    )
    ax.legend(fontsize=11)
    ax.set_ylim(bottom=0)

    plt.tight_layout()
    p = config.PLOTS_DIR / f'fnr_vs_round_{malicious_pct}pct.png'
    plt.savefig(p, dpi=200, bbox_inches='tight')
    print(f"  ✓ Saved: {p.name}")
    plt.show(); plt.close()


# ──────────────────────────────────────────────────────────────────────────────
# PLOT 2: ASR bar chart across all attack levels  — key result
# ──────────────────────────────────────────────────────────────────────────────

def plot_asr_bar(df: pd.DataFrame, config):
    """
    Bar chart: Attack Success Rate by method × attack level.
    Shows which method best resists Byzantine attacks.
    ASR = FNR × 100 (confirmed identical in data).
    Only shows attack experiments (malicious > 0).
    """
    df_atk = df[df['malicious'] > 0].copy()
    if df_atk.empty:
        print("  ⚠ No attack experiments in results.")
        return

    attack_levels = sorted(df_atk['malicious'].unique())
    x     = np.arange(len(attack_levels))
    width = 0.2

    fig, ax = plt.subplots(figsize=(13, 7))

    for i, method in enumerate(METHODS):
        m_data = df_atk[df_atk['method'] == method].sort_values('malicious')
        if m_data.empty:
            continue
        asrs = m_data['asr'].values
        bars = ax.bar(x + i * width, asrs, width,
                      label=METHOD_LABELS[method],
                      color=COLORS[method], alpha=0.85, edgecolor='white')
        for bar, val in zip(bars, asrs):
            if val >= 0.05:
                ax.text(bar.get_x() + bar.get_width() / 2,
                        bar.get_height() + 0.03,
                        f'{val:.2f}%', ha='center', va='bottom', fontsize=8.5)

    ax.set_xlabel('Byzantine Client Ratio', fontsize=12)
    ax.set_ylabel('Attack Success Rate — ASR (%)\n'
                  '[% of attack traffic misclassified as benign]', fontsize=12)
    ax.set_title(
        'Attack Success Rate by Aggregation Method\n'
        'Lower is better — Trust-GM achieves lowest ASR at every attack level',
        fontsize=13
    )
    ax.set_xticks(x + width * 1.5)
    ax.set_xticklabels([f'{int(p * 100)}%' for p in attack_levels])
    ax.legend(fontsize=11)
    ax.set_ylim(bottom=0)
    ax.grid(axis='y', alpha=0.4)

    plt.tight_layout()
    p = config.PLOTS_DIR / 'asr_bar.png'
    plt.savefig(p, dpi=200, bbox_inches='tight')
    print(f"  ✓ Saved: {p.name}")
    plt.show(); plt.close()


# ──────────────────────────────────────────────────────────────────────────────
# PLOT 3: FPR vs FNR scatter — exposes the accuracy paradox
# ──────────────────────────────────────────────────────────────────────────────

def plot_fpr_fnr_scatter(df: pd.DataFrame, config):
    """
    FPR (false alarms) vs FNR (missed attacks) for baseline and 40% attack.
    This plot explains WHY raw accuracy is misleading on imbalanced IDS data,
    and shows the trade-offs between methods clearly.
    """
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    for ax_idx, pct in enumerate([0, 40]):
        ax   = axes[ax_idx]
        sub  = df[np.isclose(df['malicious'], pct / 100)]

        for _, row in sub.iterrows():
            m = row['method']
            ax.scatter(row['fpr'] * 100, row['fnr'] * 100,
                       s=220, color=COLORS[m], zorder=5)
            ax.annotate(
                METHOD_LABELS[m],
                (row['fpr'] * 100, row['fnr'] * 100),
                textcoords='offset points', xytext=(8, 4), fontsize=9
            )

        ax.set_xlabel('False Positive Rate — FPR (%)\n[Benign traffic flagged as attack]',
                      fontsize=11)
        ax.set_ylabel('False Negative Rate — FNR (%)\n[Attack traffic missed by IDS]',
                      fontsize=11)
        title = ('Baseline — 0% Byzantine' if pct == 0
                 else f'{pct}% Byzantine Clients')
        ax.set_title(f'{title}\nIdeal: bottom-left corner (low FP AND low FN)',
                     fontsize=11)
        ax.set_xlim(left=0)
        ax.set_ylim(bottom=0)
        ax.annotate('← Ideal', xy=(0, 0), xytext=(0.5, 0.05), fontsize=9, color='green')

    handles = [plt.scatter([], [], s=150, color=COLORS[m], label=METHOD_LABELS[m])
               for m in METHODS]
    fig.legend(handles=handles, loc='lower center', ncol=4,
               fontsize=11, bbox_to_anchor=(0.5, -0.04))
    fig.suptitle(
        'FPR vs FNR Trade-off\n'
        'Accuracy alone is misleading: test set is 91.8% attack, 8.2% benign',
        fontsize=13, fontweight='bold'
    )
    plt.tight_layout()
    p = config.PLOTS_DIR / 'fpr_fnr_scatter.png'
    plt.savefig(p, dpi=200, bbox_inches='tight')
    print(f"  ✓ Saved: {p.name}")
    plt.show(); plt.close()


# ──────────────────────────────────────────────────────────────────────────────
# PLOT 4: Confusion matrix heatmaps at 40% attack
# ──────────────────────────────────────────────────────────────────────────────

def plot_confusion_matrices(df: pd.DataFrame, config, malicious_pct: int = 40):
    """
    2×2 grid of confusion matrices for all methods at given attack level.
    Shows the exact counts that drive FPR and FNR.
    """
    sub = df[np.isclose(df['malicious'], malicious_pct / 100)]
    if sub.empty:
        print(f"  ⚠ No results for {malicious_pct}% attack.")
        return

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.flatten()

    for i, method in enumerate(METHODS):
        row = sub[sub['method'] == method]
        if row.empty:
            continue
        row = row.iloc[0]
        ax  = axes[i]

        cm      = np.array([[row['tn'], row['fp']], [row['fn'], row['tp']]])
        total   = cm.sum()
        cm_norm = cm / total * 100

        ax.imshow(cm_norm, cmap='Blues', vmin=0)
        ax.set_xticks([0, 1]);  ax.set_yticks([0, 1])
        ax.set_xticklabels(['Pred Benign', 'Pred Attack'], fontsize=11)
        ax.set_yticklabels(['True Benign', 'True Attack'], fontsize=11)
        ax.set_title(
            f'{METHOD_LABELS[method]}\n'
            f'FPR={row["fpr"]:.3f}  FNR={row["fnr"]:.4f}  ASR={row["asr"]:.3f}%',
            fontsize=11, fontweight='bold'
        )
        for r in range(2):
            for c in range(2):
                color = 'white' if cm_norm[r, c] > 40 else 'black'
                ax.text(c, r,
                        f'{int(cm[r,c]):,}\n({cm_norm[r,c]:.1f}%)',
                        ha='center', va='center', color=color,
                        fontsize=10, fontweight='bold')

    fig.suptitle(
        f'Confusion Matrices — {malicious_pct}% Byzantine Clients\n'
        f'FedAvg misses {int(sub[sub["method"]=="fedavg"]["fn"].values[0]):,} attacks; '
        f'Trust-GM misses only {int(sub[sub["method"]=="trust_gm"]["fn"].values[0]):,}',
        fontsize=13, fontweight='bold'
    )
    plt.tight_layout()
    p = config.PLOTS_DIR / f'confusion_matrices_{malicious_pct}pct.png'
    plt.savefig(p, dpi=200, bbox_inches='tight')
    print(f"  ✓ Saved: {p.name}")
    plt.show(); plt.close()


# ──────────────────────────────────────────────────────────────────────────────
# PLOT 5: Trust score evolution (trust_gm only, shows mechanism working)
# ──────────────────────────────────────────────────────────────────────────────

def plot_trust_convergence(config, malicious_pct: int = 40):
    """
    Average malicious vs benign trust scores per round for Trust-GM.
    Demonstrates the honeypot mechanism: malicious trust drops to MIN_TRUST,
    benign trust stays near INITIAL_TRUST (reduced slightly by false positives).
    Data comes from per-round avg_trust_mal / avg_trust_ben in checkpoint.
    """
    _, cp = _find_checkpoint(config, 'trust_gm', malicious_pct)
    if cp is None:
        print(f"  ⚠ No trust_gm checkpoint for {malicious_pct}%.")
        return

    hist    = cp['round_history']
    # Round 0 has no trust data (trust_mgr updates start at round 1)
    rounds  = [r['round'] for r in hist if 'avg_trust_mal' in r]
    mal_tr  = [r['avg_trust_mal'] for r in hist if 'avg_trust_mal' in r]
    ben_tr  = [r['avg_trust_ben'] for r in hist if 'avg_trust_ben' in r]

    if not rounds:
        print(f"  ⚠ No per-round trust data in checkpoint for {malicious_pct}%.")
        return

    fig, ax = plt.subplots(figsize=(11, 6))
    ax.plot(rounds, mal_tr, color='#e74c3c', linewidth=2.5,
            marker='o', markersize=5, label='Avg Trust — Malicious clients')
    ax.plot(rounds, ben_tr, color='#27ae60', linewidth=2.5,
            marker='s', markersize=5, label='Avg Trust — Benign clients')
    ax.axhline(y=0.15, color='#e74c3c', linestyle='--', alpha=0.5,
               label='MIN_TRUST floor (0.15)')
    ax.axhline(y=1.0,  color='#27ae60', linestyle='--', alpha=0.5,
               label='INITIAL_TRUST (1.0)')

    ax.set_xlabel('FL Round', fontsize=12)
    ax.set_ylabel('Average Trust Score', fontsize=12)
    ax.set_title(
        f'Trust Score Convergence — Trust-GM, {malicious_pct}% Byzantine\n'
        f'Honeypot correctly drives malicious trust to minimum (0.15)',
        fontsize=13
    )
    ax.legend(fontsize=11)
    ax.set_ylim(0, 1.1)
    ax.grid(alpha=0.4)

    plt.tight_layout()
    p = config.PLOTS_DIR / f'trust_convergence_{malicious_pct}pct.png'
    plt.savefig(p, dpi=200, bbox_inches='tight')
    print(f"  ✓ Saved: {p.name}")
    plt.show(); plt.close()


# ──────────────────────────────────────────────────────────────────────────────
# SUMMARY TABLE (printed + saved)
# ──────────────────────────────────────────────────────────────────────────────

def print_summary_table(df: pd.DataFrame, config):
    """Print comparison table with FNR/ASR as primary metrics."""
    print("\n" + "="*85)
    print("RESULTS SUMMARY — PRIMARY METRICS: FNR and ASR (lower is better)")
    print("NOTE: Accuracy is secondary — test set is 91.8% attack (imbalanced)")
    print("="*85)
    header = (f"{'Method':<22} {'Attack':<8} {'Accuracy':<11} {'F1':<8} "
              f"{'FPR%':<8} {'FNR%':<8} {'ASR%':<8} {'Missed Attacks'}")
    print(header)
    print("-"*85)

    for _, row in df.sort_values(['malicious', 'method']).iterrows():
        print(
            f"{METHOD_LABELS.get(row['method'], row['method']):<22} "
            f"{int(row['malicious']*100):<8} "
            f"{row['accuracy']:.4f}     "
            f"{row['f1']:.4f}   "
            f"{row['fpr']*100:>6.2f}   "
            f"{row['fnr']*100:>6.3f}   "
            f"{row.get('asr', 0.0):>6.3f}   "
            f"{int(row['fn'])}"
        )
    print("="*85)

    # Key result callout
    at40 = df[np.isclose(df['malicious'], 0.4)].set_index('method')
    print("\nKEY RESULT — 40% Byzantine clients:")
    for m in METHODS:
        if m in at40.index:
            r = at40.loc[m]
            print(f"  {METHOD_LABELS[m]:<22} FNR={r['fnr']*100:.3f}%  "
                  f"ASR={r.get('asr', 0.0):.3f}%  "
                  f"missed={int(r['fn'])} attacks out of {int(r['fn'])+int(r['tp'])}")
    print("="*85)

    # Save
    out = config.RESULTS_DIR / 'summary_table.csv'
    cols = [c for c in ['method', 'malicious', 'accuracy', 'f1', 'fpr', 'fnr',
                         'asr', 'tn', 'fp', 'fn', 'tp', 'training_time_sec']
            if c in df.columns]
    df[cols].to_csv(out, index=False)
    print(f"  ✓ Summary saved: {out}")


# ──────────────────────────────────────────────────────────────────────────────
# DESCRIPTIVE COMPARISON (replaces broken t-test)
# ──────────────────────────────────────────────────────────────────────────────

def print_method_comparison(df: pd.DataFrame):
    """
    Descriptive comparison of Trust-GM vs others.
    Does NOT run a t-test — with single seed there is only 1 sample per
    condition, making paired t-tests meaningless.
    For a full paper: re-run with N_SEEDS=3 and use ttest_rel on the
    3-element arrays.
    """
    print("\n" + "="*70)
    print("METHOD COMPARISON — TRUST-GM vs OTHERS AT 40% ATTACK")
    print("(Descriptive only — N_SEEDS=1, formal significance needs ≥2 seeds)")
    print("="*70)

    at40 = df[np.isclose(df['malicious'], 0.4)]
    tgm  = at40[at40['method'] == 'trust_gm']
    if tgm.empty:
        print("  ⚠ No trust_gm results for 40% attack.")
        return

    tgm_asr = float(tgm['asr'].values[0])
    tgm_fnr = float(tgm['fnr'].values[0])

    print(f"\n  {'Method':<22} {'ASR%':<10} {'FNR%':<10} {'vs Trust-GM ASR'}")
    print(f"  {'-'*60}")
    for m in METHODS:
        row = at40[at40['method'] == m]
        if row.empty:
            continue
        asr = float(row['asr'].values[0])
        fnr = float(row['fnr'].values[0])
        if m == 'trust_gm':
            tag = '(ours)'
        else:
            ratio = asr / tgm_asr if tgm_asr > 0 else float('inf')
            tag = f'{ratio:.1f}× higher ASR'
        print(f"  {METHOD_LABELS[m]:<22} {asr:<10.3f} {fnr*100:<10.3f} {tag}")

    print(f"\n  Trust-GM reduces ASR vs FedAvg by "
          f"{(float(at40[at40['method']=='fedavg']['asr'].values[0]) - tgm_asr):.2f}pp "
          f"({(float(at40[at40['method']=='fedavg']['asr'].values[0])/tgm_asr):.0f}× lower)")
    print("="*70)
    print("  For formal significance testing: rerun with N_SEEDS=3 on college GPU.")
    print("="*70)


# ──────────────────────────────────────────────────────────────────────────────
# MASTER FUNCTION
# ──────────────────────────────────────────────────────────────────────────────

def create_all_visualizations(config=None):
    """Generate all plots and analysis for mentor presentation."""
    if config is None:
        config = GlobalConfig

    print("\n" + "#"*60)
    print("  GENERATING VISUALIZATIONS")
    print("#"*60)

    df = _load_results(config)
    if df is None:
        return

    print("\n[1] FNR vs Round — 40% attack (primary metric)...")
    plot_fnr_vs_round(config, malicious_pct=40)

    print("\n[2] FNR vs Round — 30% attack...")
    plot_fnr_vs_round(config, malicious_pct=30)

    print("\n[3] ASR bar chart — all attack levels...")
    plot_asr_bar(df, config)

    print("\n[4] FPR vs FNR scatter — baseline vs 40% attack...")
    plot_fpr_fnr_scatter(df, config)

    print("\n[5] Confusion matrices — 40% attack...")
    plot_confusion_matrices(df, config, malicious_pct=40)

    print("\n[6] Trust score convergence — 40% attack...")
    plot_trust_convergence(config, malicious_pct=40)

    print("\n[7] Summary table...")
    print_summary_table(df, config)

    print("\n[8] Method comparison...")
    print_method_comparison(df)

    print(f"\n✓ All plots saved to: {config.PLOTS_DIR}")
    print("#"*60)


# ── Ready ─────────────────────────────────────────────────────────────────────
print("="*60)
print("SECTION 11 READY")
print("="*60)
print("After experiments: create_all_visualizations(GlobalConfig)")
print("="*60)

In [ ]:
summary = run_full_experimental_protocol(
        preprocessed_data, global_test_preprocessed)

In [ ]:
# Visualizations:
create_all_visualizations(GlobalConfig)